In [10]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Set, Optional, Union
import pandas as pd
import requests
import os
import xml.etree.ElementTree as ET
import obonet  # pip install obonet
from tqdm import tqdm  # For progress bars

class ChEBIOntologyReasoner:
    def __init__(self, load_from_file: bool = True, file_path: str = "chebi.obo"):
        """
        Initialize the ChEBI ontology reasoner
        
        Args:
            load_from_file: Whether to load the ontology from a local file
            file_path: Path to the ChEBI ontology file (.obo format)
        """
        # Initialize the ontology as a directed graph
        self.ontology = nx.MultiDiGraph()
        
        # Store rules and relationships
        self.rules = []
        self.relationships = {}
        
        # Track IDs to names mapping
        self.id_to_name = {}
        self.name_to_id = {}
        
        # Load ChEBI ontology if specified
        if load_from_file:
            self.load_chebi_ontology(file_path)
    
    def load_chebi_ontology(self, file_path: str):
        """
        Load the ChEBI ontology from an OBO file
        
        Args:
            file_path: Path to the ChEBI ontology file (.obo format)
        """
        # Check if file exists, download if not
        if not os.path.exists(file_path):
            print(f"ChEBI ontology file not found at {file_path}. Downloading...")
            self._download_chebi_ontology(file_path)
        
        print(f"Loading ChEBI ontology from {file_path}...")
        # Load the ontology using obonet
        try:
            graph = obonet.read_obo(file_path)
            
            # Convert obonet graph to networkx MultiDiGraph
            for node_id, data in tqdm(graph.nodes(data=True), desc="Loading nodes"):
                # Extract entity name from data
                name = data.get('name', node_id)
                
                # Store ID to name mapping
                self.id_to_name[node_id] = name
                self.name_to_id[name] = node_id
                
                # Add node to the ontology - ensure we don't duplicate 'name' attribute
                node_attrs = {k: v for k, v in data.items() if k != 'name'}
                node_attrs['entity_name'] = name  # Use 'entity_name' instead of 'name' to avoid conflicts
                self.ontology.add_node(node_id, **node_attrs)
            
            # Add relationships
            for u, v, key, data in tqdm(graph.edges(keys=True, data=True), desc="Loading relationships"):
                relation_type = key
                
                # Add edge to the ontology
                self.ontology.add_edge(u, v, relation=relation_type, **data)
                
                # Store relationship for easier access
                if relation_type not in self.relationships:
                    self.relationships[relation_type] = []
                self.relationships[relation_type].append((u, v))
            
            print(f"ChEBI ontology loaded successfully with {self.ontology.number_of_nodes()} nodes and {self.ontology.number_of_edges()} edges.")
        
        except Exception as e:
            print(f"Error loading ChEBI ontology: {e}")
            raise
    
    def _download_chebi_ontology(self, file_path: str):
        """
        Download the ChEBI ontology from the official repository
        
        Args:
            file_path: Where to save the downloaded ontology
        """
        url = "https://ftp.ebi.ac.uk/pub/databases/chebi/ontology/chebi.obo"
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            block_size = 8192
            
            with open(file_path, 'wb') as f:
                for chunk in tqdm(
                    response.iter_content(chunk_size=block_size),
                    total=total_size//block_size,
                    unit='KB',
                    desc="Downloading ChEBI ontology"
                ):
                    if chunk:
                        f.write(chunk)
            
            print(f"ChEBI ontology downloaded successfully to {file_path}")
        
        except Exception as e:
            print(f"Error downloading ChEBI ontology: {e}")
            if os.path.exists(file_path):
                os.remove(file_path)
            raise
    
    def add_concept(self, concept_id: str, properties: Dict = None):
        """Add a concept to the ontology with optional properties"""
        if properties is None:
            properties = {}
        
        # If concept_id is a name, convert to ID if possible
        if concept_id in self.name_to_id:
            concept_id = self.name_to_id[concept_id]
        
        # Use 'entity_name' instead of 'name' to avoid conflicts
        if 'name' in properties:
            properties['entity_name'] = properties.pop('name')
            
        self.ontology.add_node(concept_id, **properties)
    
    def add_relationship(self, source: str, target: str, relation_type: str, properties: Dict = None):
        """
        Add a relationship between two concepts
        
        Args:
            source: Source concept ID or name
            target: Target concept ID or name
            relation_type: Type of relation
            properties: Optional properties for the relationship
        """
        if properties is None:
            properties = {}
        
        # Convert names to IDs if possible
        if source in self.name_to_id:
            source = self.name_to_id[source]
        if target in self.name_to_id:
            target = self.name_to_id[target]
        
        # Add edge to the ontology
        self.ontology.add_edge(source, target, relation=relation_type, **properties)
        
        # Store relationship for easier access
        if relation_type not in self.relationships:
            self.relationships[relation_type] = []
        self.relationships[relation_type].append((source, target))
    
    def add_rule(self, condition: str, consequence: str, description: str = None):
        """Add a rule to the ontology with optional description"""
        self.rules.append((condition, consequence, description))
    
    def get_entity_by_id(self, entity_id: str) -> Dict:
        """
        Get entity data by ID
        
        Args:
            entity_id: ChEBI ID
        
        Returns:
            Dictionary with entity properties
        """
        if entity_id in self.ontology.nodes:
            return self.ontology.nodes[entity_id]
        return None
    
    def get_entity_by_name(self, name: str) -> Dict:
        """
        Get entity data by name
        
        Args:
            name: Entity name
        
        Returns:
            Dictionary with entity properties
        """
        if name in self.name_to_id:
            entity_id = self.name_to_id[name]
            return self.get_entity_by_id(entity_id)
        return None
    
    def search_entities(self, query: str, limit: int = 10) -> List[Tuple[str, str]]:
        """
        Search for entities by name or ID
        
        Args:
            query: Search query
            limit: Maximum number of results
        
        Returns:
            List of tuples containing (entity_id, entity_name)
        """
        results = []
        
        # Search by ID
        if query.startswith("CHEBI:"):
            if query in self.ontology.nodes:
                name = self.id_to_name.get(query, query)
                results.append((query, name))
        
        # Search by name
        query_lower = query.lower()
        for entity_id, name in self.id_to_name.items():
            if query_lower in name.lower():
                results.append((entity_id, name))
                if len(results) >= limit:
                    break
        
        return results
    
    def get_related_concepts(self, concept: str, relation_type: str = None) -> List[Tuple[str, str, str]]:
        """
        Get all concepts related to a given concept, optionally filtered by relation type
        
        Args:
            concept: Concept ID or name
            relation_type: Optional relation type filter
        
        Returns:
            List of tuples containing (source, target, relation_type)
        """
        # Convert name to ID if possible
        if concept in self.name_to_id:
            concept = self.name_to_id[concept]
        
        related = []
        
        # Get outgoing relationships
        for _, target, data in self.ontology.out_edges(concept, data=True):
            edge_relation = data.get('relation', '')
            if relation_type is None or edge_relation == relation_type:
                source_name = self.id_to_name.get(concept, concept)
                target_name = self.id_to_name.get(target, target)
                related.append((source_name, target_name, edge_relation))
        
        # Get incoming relationships
        for source, _, data in self.ontology.in_edges(concept, data=True):
            edge_relation = data.get('relation', '')
            if relation_type is None or edge_relation == relation_type:
                source_name = self.id_to_name.get(source, source)
                target_name = self.id_to_name.get(concept, concept)
                related.append((source_name, target_name, edge_relation))
        
        return related
    
    def get_concepts_with_property(self, property_name: str, property_value=None) -> List[str]:
        """
        Get all concepts with a specific property
        
        Args:
            property_name: Name of the property
            property_value: Optional property value filter
        
        Returns:
            List of concept names
        """
        concepts = []
        for node, properties in self.ontology.nodes(data=True):
            if property_name in properties:
                if property_value is None or properties[property_name] == property_value:
                    concepts.append(self.id_to_name.get(node, node))
        return concepts
    
    def get_ancestors(self, concept: str, relation_type: str = "is_a") -> List[str]:
        """
        Get all ancestors of a concept following a specific relationship type
        
        Args:
            concept: Concept ID or name
            relation_type: Relation type to follow
        
        Returns:
            List of ancestor concept names
        """
        # Convert name to ID if possible
        if concept in self.name_to_id:
            concept = self.name_to_id[concept]
        
        ancestors = []
        
        # Define recursive function to traverse ancestors
        def get_ancestors_recursive(node, relation, visited=None):
            if visited is None:
                visited = set()
            
            if node in visited:
                return
            
            visited.add(node)
            
            for _, parent, data in self.ontology.out_edges(node, data=True):
                edge_relation = data.get('relation', '')
                if edge_relation == relation:
                    parent_name = self.id_to_name.get(parent, parent)
                    ancestors.append(parent_name)
                    get_ancestors_recursive(parent, relation, visited)
        
        get_ancestors_recursive(concept, relation_type)
        return ancestors
    
    def get_descendants(self, concept: str, relation_type: str = "is_a") -> List[str]:
        """
        Get all descendants of a concept following a specific relationship type
        
        Args:
            concept: Concept ID or name
            relation_type: Relation type to follow
        
        Returns:
            List of descendant concept names
        """
        # Convert name to ID if possible
        if concept in self.name_to_id:
            concept = self.name_to_id[concept]
        
        descendants = []
        
        # Define recursive function to traverse descendants
        def get_descendants_recursive(node, relation, visited=None):
            if visited is None:
                visited = set()
            
            if node in visited:
                return
            
            visited.add(node)
            
            for child, _, data in self.ontology.in_edges(node, data=True):
                edge_relation = data.get('relation', '')
                if edge_relation == relation:
                    child_name = self.id_to_name.get(child, child)
                    descendants.append(child_name)
                    get_descendants_recursive(child, relation, visited)
        
        get_descendants_recursive(concept, relation_type)
        return descendants
    
    def apply_rules(self, facts: Set[str]) -> Tuple[Set[str], List[str]]:
        """
        Apply rules to derive new facts and return reasoning steps
        
        Args:
            facts: Set of initial facts
        
        Returns:
            Tuple containing (updated facts, reasoning steps)
        """
        new_facts = facts.copy()
        reasoning_steps = []
        
        change_made = True
        while change_made:
            change_made = False
            for condition, consequence, description in self.rules:
                if condition in new_facts and consequence not in new_facts:
                    new_facts.add(consequence)
                    change_made = True
                    if description:
                        reasoning_steps.append(f"Applied rule: {description}")
                    else:
                        reasoning_steps.append(f"From '{condition}', derived '{consequence}'")
        
        return new_facts, reasoning_steps
    
    def _extract_entities_from_question(self, question: str) -> List[str]:
        """
        Extract chemical entity names from a question
        
        Args:
            question: Question to extract entities from
        
        Returns:
            List of entity names found in the question
        """
        entities = []
        
        # Simple extraction based on known entities
        for name in self.name_to_id.keys():
            if name in question:
                entities.append(name)
        
        # If specific ChEBI IDs are mentioned
        if "CHEBI:" in question:
            import re
            chebi_ids = re.findall(r'CHEBI:\d+', question)
            for chebi_id in chebi_ids:
                if chebi_id in self.id_to_name:
                    entities.append(self.id_to_name[chebi_id])
        
        return entities
    
    def reason(self, question: str) -> List[str]:
        """
        Generate reasoning steps from a question
        
        Args:
            question: Question to reason about
        
        Returns:
            List of reasoning steps
        """
        reasoning_steps = ["Question: " + question]
        
        # Extract entities from question
        entities = self._extract_entities_from_question(question)
        
        if entities:
            reasoning_steps.append(f"Step 1: Identified entities: {', '.join(entities)}")
            
            # Lookup entities in the ontology
            entity_info = []
            for entity in entities:
                entity_data = self.get_entity_by_name(entity)
                if entity_data:
                    entity_info.append(f"{entity} (Found in ChEBI)")
                else:
                    entity_info.append(f"{entity} (Not found in ChEBI)")
            
            reasoning_steps.append(f"Step 2: Entity lookup: {', '.join(entity_info)}")
            
            # Process question patterns
            if "classification" in question.lower() or "type" in question.lower():
                self._reason_classification(entities, reasoning_steps)
            
            elif "relationship" in question.lower() or "related" in question.lower():
                self._reason_relationships(entities, reasoning_steps)
            
            elif "property" in question.lower() or "properties" in question.lower():
                self._reason_properties(entities, reasoning_steps)
            
            elif "pathway" in question.lower() or "reaction" in question.lower():
                self._reason_pathways(entities, reasoning_steps)
            
            else:
                reasoning_steps.append("Step 3: Providing general information about entities")
                for entity in entities:
                    if entity in self.name_to_id:
                        entity_id = self.name_to_id[entity]
                        ancestors = self.get_ancestors(entity_id)
                        if ancestors:
                            reasoning_steps.append(f"{entity} is classified as: {', '.join(ancestors[:5])}")
                        
                        related = self.get_related_concepts(entity_id)
                        if related:
                            related_info = [f"{source} {relation} {target}" for source, target, relation in related[:5]]
                            reasoning_steps.append(f"Related concepts: {'; '.join(related_info)}")
        else:
            reasoning_steps.append("No specific chemical entities were identified in the question.")
        
        return reasoning_steps
    
    def _reason_classification(self, entities: List[str], reasoning_steps: List[str]):
        """
        Generate reasoning steps for classification questions
        
        Args:
            entities: List of entity names
            reasoning_steps: List to append reasoning steps to
        """
        reasoning_steps.append("Step 3: Analyzing classification hierarchy")
        
        for entity in entities:
            if entity in self.name_to_id:
                entity_id = self.name_to_id[entity]
                
                # Get ancestors (classifications)
                ancestors = self.get_ancestors(entity_id)
                if ancestors:
                    reasoning_steps.append(f"{entity} is classified as: {', '.join(ancestors[:10])}")
                else:
                    reasoning_steps.append(f"{entity} has no parent classifications")
                
                # Get descendants (subclasses)
                descendants = self.get_descendants(entity_id)
                if descendants:
                    reasoning_steps.append(f"Subclasses of {entity}: {', '.join(descendants[:10])}")
                    if len(descendants) > 10:
                        reasoning_steps.append(f"...and {len(descendants) - 10} more")
                else:
                    reasoning_steps.append(f"{entity} has no subclasses")
    
    def _reason_relationships(self, entities: List[str], reasoning_steps: List[str]):
        """
        Generate reasoning steps for relationship questions
        
        Args:
            entities: List of entity names
            reasoning_steps: List to append reasoning steps to
        """
        reasoning_steps.append("Step 3: Analyzing relationships between entities")
        
        if len(entities) >= 2:
            # Look for direct relationships between pairs of entities
            for i in range(len(entities)):
                for j in range(i+1, len(entities)):
                    entity1, entity2 = entities[i], entities[j]
                    
                    if entity1 in self.name_to_id and entity2 in self.name_to_id:
                        entity1_id = self.name_to_id[entity1]
                        entity2_id = self.name_to_id[entity2]
                        
                        # Check for direct relationships
                        direct_relations = []
                        
                        # Get edges from entity1 to entity2
                        edge_data = self.ontology.get_edge_data(entity1_id, entity2_id)
                        if edge_data:
                            for key, data in edge_data.items():
                                relation = data.get('relation', 'related_to')
                                direct_relations.append(f"{entity1} {relation} {entity2}")
                        
                        # Get edges from entity2 to entity1
                        edge_data = self.ontology.get_edge_data(entity2_id, entity1_id)
                        if edge_data:
                            for key, data in edge_data.items():
                                relation = data.get('relation', 'related_to')
                                direct_relations.append(f"{entity2} {relation} {entity1}")
                        
                        if direct_relations:
                            reasoning_steps.append(f"Direct relationships: {'; '.join(direct_relations)}")
                        else:
                            reasoning_steps.append(f"No direct relationships found between {entity1} and {entity2}")
                            
                            # Check for common ancestors
                            ancestors1 = set(self.get_ancestors(entity1_id))
                            ancestors2 = set(self.get_ancestors(entity2_id))
                            common_ancestors = ancestors1.intersection(ancestors2)
                            
                            if common_ancestors:
                                reasoning_steps.append(f"Common classifications: {', '.join(list(common_ancestors)[:5])}")
        
        # For each entity, show its important relationships
        for entity in entities:
            if entity in self.name_to_id:
                entity_id = self.name_to_id[entity]
                related = self.get_related_concepts(entity_id)
                
                # Group by relation type
                by_relation = {}
                for source, target, relation in related:
                    if relation not in by_relation:
                        by_relation[relation] = []
                    
                    if source == entity:
                        by_relation[relation].append(target)
                    else:
                        by_relation[relation].append(source)
                
                # Show important relations
                for relation, related_entities in by_relation.items():
                    if len(related_entities) > 0:
                        reasoning_steps.append(f"{entity} has {relation} relationship with: {', '.join(related_entities[:5])}")
                        if len(related_entities) > 5:
                            reasoning_steps.append(f"...and {len(related_entities) - 5} more")
    
    def _reason_properties(self, entities: List[str], reasoning_steps: List[str]):
        """
        Generate reasoning steps for property questions
        
        Args:
            entities: List of entity names
            reasoning_steps: List to append reasoning steps to
        """
        reasoning_steps.append("Step 3: Analyzing properties of entities")
        
        for entity in entities:
            if entity in self.name_to_id:
                entity_id = self.name_to_id[entity]
                entity_data = self.get_entity_by_id(entity_id)
                
                if entity_data:
                    # Extract properties
                    properties = []
                    for key, value in entity_data.items():
                        if key != 'entity_name' and not key.startswith('_'):
                            properties.append(f"{key}: {value}")
                    
                    if properties:
                        reasoning_steps.append(f"Properties of {entity}:")
                        for prop in properties:
                            reasoning_steps.append(f"- {prop}")
                    else:
                        reasoning_steps.append(f"No properties found for {entity}")
                
                # Look for specific ChEBI properties like mass, formula, etc.
                for relation in ["has_mass", "has_formula", "has_charge"]:
                    related = self.get_related_concepts(entity_id, relation)
                    if related:
                        for source, target, rel in related:
                            reasoning_steps.append(f"{entity} {rel}: {target}")
    
    def _reason_pathways(self, entities: List[str], reasoning_steps: List[str]):
        """
        Generate reasoning steps for pathway/reaction questions
        
        Args:
            entities: List of entity names
            reasoning_steps: List to append reasoning steps to
        """
        reasoning_steps.append("Step 3: Analyzing pathways and reactions")
        
        for entity in entities:
            if entity in self.name_to_id:
                entity_id = self.name_to_id[entity]
                
                # Look for participation in reactions
                reactions = []
                
                # Look for 'is_reactant_in', 'is_product_of' relationships
                for relation in ["is_reactant_in", "is_product_of", "is_substrate_of", "is_inhibitor_of"]:
                    related = self.get_related_concepts(entity_id, relation)
                    for source, target, rel in related:
                        reactions.append(f"{entity} {rel} {target}")
                
                if reactions:
                    reasoning_steps.append(f"Reaction participation for {entity}:")
                    for reaction in reactions:
                        reasoning_steps.append(f"- {reaction}")
                else:
                    reasoning_steps.append(f"No reaction participation found for {entity}")
    
    def visualize(self, concept: str = None, max_nodes: int = 50, relation_types: List[str] = None):
        """
        Visualize the ontology or a subset around a concept
        
        Args:
            concept: Optional center concept to visualize neighborhood
            max_nodes: Maximum number of nodes to include
            relation_types: Optional list of relation types to include
        """
        plt.figure(figsize=(14, 10))
        
        # Create subgraph to visualize
        if concept:
            # Convert name to ID if possible
            if concept in self.name_to_id:
                concept = self.name_to_id[concept]
            
            # Create neighborhood subgraph
            nodes = set([concept])
            edges = []
            
            # Add immediate neighbors
            for source, target, data in self.ontology.out_edges(concept, data=True):
                rel = data.get('relation', '')
                if relation_types is None or rel in relation_types:
                    nodes.add(target)
                    edges.append((source, target, data))
            
            for source, target, data in self.ontology.in_edges(concept, data=True):
                rel = data.get('relation', '')
                if relation_types is None or rel in relation_types:
                    nodes.add(source)
                    edges.append((source, target, data))
            
            # If we have room, add second-level neighbors
            if len(nodes) < max_nodes:
                additional_nodes = set()
                additional_edges = []
                
                for node in list(nodes):
                    if node != concept:  # Skip center node
                        for source, target, data in self.ontology.out_edges(node, data=True):
                            rel = data.get('relation', '')
                            if (relation_types is None or rel in relation_types) and target not in nodes:
                                additional_nodes.add(target)
                                additional_edges.append((source, target, data))
                                if len(nodes) + len(additional_nodes) >= max_nodes:
                                    break
                        
                        for source, target, data in self.ontology.in_edges(node, data=True):
                            rel = data.get('relation', '')
                            if (relation_types is None or rel in relation_types) and source not in nodes:
                                additional_nodes.add(source)
                                additional_edges.append((source, target, data))
                                if len(nodes) + len(additional_nodes) >= max_nodes:
                                    break
                
                nodes.update(additional_nodes)
                edges.extend(additional_edges)
            
            # Create subgraph
            subgraph = nx.MultiDiGraph()
            for node in nodes:
                node_data = self.ontology.nodes[node]
                name = self.id_to_name.get(node, node)
                # Use entity_name instead of name to avoid conflicts
                node_data_copy = {k: v for k, v in node_data.items() if k != 'name'}
                subgraph.add_node(node, entity_name=name, **node_data_copy)
            
            for source, target, data in edges:
                subgraph.add_edge(source, target, **data)
            
            graph = subgraph
        else:
            # Limit to max_nodes
            node_list = list(self.ontology.nodes())[:max_nodes]
            graph = self.ontology.subgraph(node_list)
        
        # Create position layout
        pos = nx.spring_layout(graph, k=0.8, iterations=50)
        
        # Draw nodes
        nx.draw_networkx_nodes(graph, pos, node_size=1800, node_color="lightblue")
        
        # Draw node labels
        labels = {}
        for node in graph.nodes():
            name = self.id_to_name.get(node, node)
            # Truncate long names
            if len(name) > 20:
                name = name[:17] + "..."
            labels[node] = name
        
        nx.draw_networkx_labels(graph, pos, labels=labels, font_size=10)
        
        # Draw edges with different colors based on relation type
        edge_colors = {}
        for i, rel_type in enumerate(set(data.get('relation', '') for _, _, data in graph.edges(data=True))):
            # Generate a color based on relation type
            color = plt.cm.tab10(i % 10)
            edge_colors[rel_type] = color
        
        # Draw edges grouped by relation type
        for rel_type, color in edge_colors.items():
            edges = [(s, t) for s, t, d in graph.edges(data=True) if d.get('relation', '') == rel_type]
            if edges:
                nx.draw_networkx_edges(graph, pos, edgelist=edges, 
                                      edge_color=[color]*len(edges), width=1.5, 
                                      label=rel_type, connectionstyle='arc3,rad=0.1')
        
        plt.title("ChEBI Ontology Visualization")
        if len(edge_colors) > 0:
            plt.legend()
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    def export_subgraph(self, concept: str, depth: int = 2, relation_types: List[str] = None, output_file: str = "subgraph.graphml"):
        """
        Export a subgraph around a concept to a GraphML file
        
        Args:
            concept: Center concept
            depth: Neighborhood depth
            relation_types: Optional list of relation types to include
            output_file: Output GraphML file path
        """
        # Convert name to ID if possible
        if concept in self.name_to_id:
            concept = self.name_to_id[concept]
        
        # Create subgraph
        subgraph = nx.MultiDiGraph()
        
        # Add center node
        if concept in self.ontology.nodes:
            node_data = self.ontology.nodes[concept]
            name = self.id_to_name.get(concept, concept)
            
            # Just add the node directly - avoid any attribute manipulation
            subgraph.add_node(concept)
            
            # Then add all attributes manually to avoid conflicts
            for key, value in node_data.items():
                if key != 'entity_name':  # Skip existing entity_name if present
                    subgraph.nodes[concept][key] = value
            
            # Set the entity_name attribute
            subgraph.nodes[concept]['entity_name'] = name
            
            # BFS to get neighborhood
            visited = {concept}
            current_level = {concept}
            
            for _ in range(depth):
                next_level = set()
                
                for node in current_level:
                    # Outgoing edges
                    for source, target, data in self.ontology.out_edges(node, data=True):
                        rel = data.get('relation', '')
                        if relation_types is None or rel in relation_types:
                            if target not in subgraph:
                                # Add node first
                                subgraph.add_node(target)
                                
                                # Then add attributes
                                target_data = self.ontology.nodes[target]
                                target_name = self.id_to_name.get(target, target)
                                
                                for key, value in target_data.items():
                                    if key != 'entity_name':  # Skip existing entity_name if present
                                        subgraph.nodes[target][key] = value
                                
                                # Set the entity_name attribute
                                subgraph.nodes[target]['entity_name'] = target_name
                            
                            # Add edge with all attributes
                            subgraph.add_edge(source, target, **data)
                            
                            if target not in visited:
                                next_level.add(target)
                                visited.add(target)
                    
                    # Incoming edges
                    for source, target, data in self.ontology.in_edges(node, data=True):
                        rel = data.get('relation', '')
                        if relation_types is None or rel in relation_types:
                            if source not in subgraph:
                                # Add node first
                                subgraph.add_node(source)
                                
                                # Then add attributes
                                source_data = self.ontology.nodes[source]
                                source_name = self.id_to_name.get(source, source)
                                
                                for key, value in source_data.items():
                                    if key != 'entity_name':  # Skip existing entity_name if present
                                        subgraph.nodes[source][key] = value
                                
                                # Set the entity_name attribute
                                subgraph.nodes[source]['entity_name'] = source_name
                            
                            # Add edge with all attributes
                            subgraph.add_edge(source, target, **data)
                            
                            if source not in visited:
                                next_level.add(source)
                                visited.add(source)
                
                current_level = next_level
                if not current_level:
                    break
            
            # Export to GraphML
            try:
                nx.write_graphml(subgraph, output_file)
                print(f"Subgraph exported to {output_file} with {subgraph.number_of_nodes()} nodes and {subgraph.number_of_edges()} edges")
            except Exception as e:
                print(f"Error exporting subgraph: {e}")
        else:
            print(f"Concept {concept} not found in the ontology")

# Simple demonstration function to test the reasoner
def demonstrate_chebi_reasoner():
    """Demonstrate the basic functionality of the ChEBI ontology reasoner"""
    print("Initializing ChEBI Ontology Reasoner...")
    reasoner = ChEBIOntologyReasoner(load_from_file=True)
    
    # Example 1: Search for an entity
    print("\n=== Example 1: Search for an entity ===")
    search_term = "dopamine"
    results = reasoner.search_entities(search_term)
    
    if results:
        print(f"Found {len(results)} results for '{search_term}':")
        for entity_id, entity_name in results[:5]:
            print(f"  - {entity_name} ({entity_id})")
        
        # Get the first result for further examples
        entity_id, entity_name = results[0]
        
        # Example 2: Get entity classifications
        print(f"\n=== Example 2: Classifications for {entity_name} ===")
        ancestors = reasoner.get_ancestors(entity_id)
        print(f"Classifications (is_a relationships):")
        for i, ancestor in enumerate(ancestors[:10]):
            print(f"  {i+1}. {ancestor}")
        
        # Example 3: Get related concepts
        print(f"\n=== Example 3: Concepts related to {entity_name} ===")
        related = reasoner.get_related_concepts(entity_id)
        print(f"Related concepts:")
        for i, (source, target, relation) in enumerate(related[:10]):
            print(f"  {i+1}. {source} {relation} {target}")
        
        # Example 4: Get descendants (subclasses)
        print(f"\n=== Example 4: Subclasses of {entity_name} ===")
        descendants = reasoner.get_descendants(entity_id)
        print(f"Subclasses:")
        for i, descendant in enumerate(descendants[:10]):
            print(f"  {i+1}. {descendant}")
        
        # Example 5: Export a subgraph
        print(f"\n=== Example 5: Export subgraph for {entity_name} ===")
        output_file = f"{entity_name.replace(' ', '_')}_subgraph.graphml"
        print(f"Exporting subgraph to {output_file}")
        reasoner.export_subgraph(entity_id, depth=1, relation_types=["is_a", "has_role", "has_part"], output_file=output_file)
    else:
        print(f"No results found for '{search_term}'")


if __name__ == "__main__":
    demonstrate_chebi_reasoner()

Initializing ChEBI Ontology Reasoner...
Loading ChEBI ontology from chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:00<00:00, 559152.49it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

=== Example 1: Search for an entity ===
Found 10 results for 'dopamine':
  - dopamine quinone(1+) (CHEBI:167191)
  - 3-O-Methyl-a-methyldopamine (CHEBI:173516)
  - a-Methyldopamine (CHEBI:193722)
  - oxidopamine (CHEBI:78741)
  - dopamine 3-O-sulfate (CHEBI:37946)

=== Example 2: Classifications for dopamine quinone(1+) ===
Classifications (is_a relationships):
  1. primary ammonium ion
  2. ammonium ion derivative
  3. nitrogen molecular entity
  4. pnictogen molecular entity
  5. p-block molecular entity
  6. main group molecular entity
  7. molecular entity
  8. chemical entity
  9. polyatomic cation
  10. cation

=== Example 3: Concepts related to dopamine quinone(1+) ===
Related concepts:
  1. dopamine quinone(1+) is_a primary ammonium ion
  2. dopamine quinone(1+) is_conjugate_acid_of dopaminoquinone
  3. dopaminoquinone is_conjugate_base_of dopamine quinone(1+)

=== Example 4: Subclasses of dopamine quinone(

In [11]:


def test_chebi_reasoner():
    """Test the basic functionality of the ChEBI Ontology Reasoner"""
    # Create data directory if it doesn't exist
    os.makedirs("data", exist_ok=True)
    
    # Initialize reasoner with the path to save the ontology file
    print("Initializing ChEBI Ontology Reasoner...")
    reasoner = ChEBIOntologyReasoner(load_from_file=True, file_path="data/chebi.obo")
    
    # Test 1: Basic entity search
    test_entity = "dopamine"
    print(f"\nTest 1: Searching for '{test_entity}'")
    results = reasoner.search_entities(test_entity)
    
    if results:
        print(f"Found {len(results)} results:")
        for entity_id, entity_name in results[:5]:
            print(f"  - {entity_name} ({entity_id})")
        
        # Get first result for additional tests
        entity_id, entity_name = results[0]
        
        # Test 2: Get classifications
        print(f"\nTest 2: Getting classifications for '{entity_name}'")
        ancestors = reasoner.get_ancestors(entity_id)
        print(f"Found {len(ancestors)} classifications:")
        for i, ancestor in enumerate(ancestors[:5]):
            print(f"  {i+1}. {ancestor}")
        
        # Test 3: Get related concepts
        print(f"\nTest 3: Getting concepts related to '{entity_name}'")
        related = reasoner.get_related_concepts(entity_id)
        print(f"Found {len(related)} related concepts:")
        for i, (source, target, relation) in enumerate(related[:5]):
            print(f"  {i+1}. {source} {relation} {target}")
        
        # Test 4: Reasoning about a question
        print(f"\nTest 4: Reasoning about a question involving '{entity_name}'")
        question = f"What is the classification of {entity_name}?"
        reasoning_steps = reasoner.reason(question)
        print("Reasoning steps:")
        for step in reasoning_steps:
            print(f"  {step}")
        
        # Test 5: Export subgraph
        print(f"\nTest 5: Exporting subgraph for '{entity_name}'")
        output_dir = "output"
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f"{entity_name.replace(' ', '_')}_subgraph.graphml")
        reasoner.export_subgraph(entity_id, depth=1, output_file=output_file)
    else:
        print(f"No results found for '{test_entity}'")

if __name__ == "__main__":
    test_chebi_reasoner()

Initializing ChEBI Ontology Reasoner...
ChEBI ontology file not found at data/chebi.obo. Downloading...


ChEBI ontology downloaded successfully to data/chebi.obo
Loading ChEBI ontology from data/chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:01<00:00, 210008.64it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

Test 1: Searching for 'dopamine'
Found 10 results:
  - dopamine quinone(1+) (CHEBI:167191)
  - 3-O-Methyl-a-methyldopamine (CHEBI:173516)
  - a-Methyldopamine (CHEBI:193722)
  - oxidopamine (CHEBI:78741)
  - dopamine 3-O-sulfate (CHEBI:37946)

Test 2: Getting classifications for 'dopamine quinone(1+)'
Found 23 classifications:
  1. primary ammonium ion
  2. ammonium ion derivative
  3. nitrogen molecular entity
  4. pnictogen molecular entity
  5. p-block molecular entity

Test 3: Getting concepts related to 'dopamine quinone(1+)'
Found 3 related concepts:
  1. dopamine quinone(1+) is_a primary ammonium ion
  2. dopamine quinone(1+) is_conjugate_acid_of dopaminoquinone
  3. dopaminoquinone is_conjugate_base_of dopamine quinone(1+)

Test 4: Reasoning about a question involving 'dopamine quinone(1+)'
Reasoning steps:
  Question: What is the classification of dopamine quinone(1+)?
  Step 1: Identified entities: ion, c

In [12]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Set, Optional
import requests
import os
import obonet  # pip install obonet
from tqdm import tqdm  # For progress bars

class BasicChEBIReasoner:
    """
    A simplified ChEBI ontology reasoner focusing on basic functionality
    """
    
    def __init__(self, load_from_file: bool = True, file_path: str = "chebi.obo"):
        """
        Initialize the ChEBI ontology reasoner
        
        Args:
            load_from_file: Whether to load the ontology from a local file
            file_path: Path to the ChEBI ontology file (.obo format)
        """
        # Initialize the ontology as a directed graph
        self.ontology = nx.MultiDiGraph()
        
        # Track IDs to names mapping
        self.id_to_name = {}
        self.name_to_id = {}
        
        # Load ChEBI ontology if specified
        if load_from_file:
            self.load_chebi_ontology(file_path)
    
    def load_chebi_ontology(self, file_path: str):
        """
        Load the ChEBI ontology from an OBO file
        
        Args:
            file_path: Path to the ChEBI ontology file (.obo format)
        """
        # Check if file exists, download if not
        if not os.path.exists(file_path):
            print(f"ChEBI ontology file not found at {file_path}. Downloading...")
            self._download_chebi_ontology(file_path)
        
        print(f"Loading ChEBI ontology from {file_path}...")
        # Load the ontology using obonet
        try:
            graph = obonet.read_obo(file_path)
            
            # Convert obonet graph to networkx MultiDiGraph
            for node_id, data in tqdm(graph.nodes(data=True), desc="Loading nodes"):
                # Extract entity name from data
                name = data.get('name', node_id)
                
                # Store ID to name mapping
                self.id_to_name[node_id] = name
                self.name_to_id[name] = node_id
                
                # Add node to the ontology - ensure we don't duplicate 'name' attribute
                node_attrs = {k: v for k, v in data.items() if k != 'name'}
                node_attrs['entity_name'] = name  # Use 'entity_name' instead of 'name'
                self.ontology.add_node(node_id, **node_attrs)
            
            # Add relationships
            for u, v, key, data in tqdm(graph.edges(keys=True, data=True), desc="Loading relationships"):
                relation_type = key
                
                # Add edge to the ontology
                self.ontology.add_edge(u, v, relation=relation_type, **data)
            
            print(f"ChEBI ontology loaded successfully with {self.ontology.number_of_nodes()} nodes and {self.ontology.number_of_edges()} edges.")
        
        except Exception as e:
            print(f"Error loading ChEBI ontology: {e}")
            raise
    
    def _download_chebi_ontology(self, file_path: str):
        """
        Download the ChEBI ontology from the official repository
        
        Args:
            file_path: Where to save the downloaded ontology
        """
        url = "https://ftp.ebi.ac.uk/pub/databases/chebi/ontology/chebi.obo"
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            block_size = 8192
            
            with open(file_path, 'wb') as f:
                for chunk in tqdm(
                    response.iter_content(chunk_size=block_size),
                    total=total_size//block_size,
                    unit='KB',
                    desc="Downloading ChEBI ontology"
                ):
                    if chunk:
                        f.write(chunk)
            
            print(f"ChEBI ontology downloaded successfully to {file_path}")
        
        except Exception as e:
            print(f"Error downloading ChEBI ontology: {e}")
            if os.path.exists(file_path):
                os.remove(file_path)
            raise
    
    def search_entities(self, query: str, limit: int = 10) -> List[Tuple[str, str]]:
        """
        Search for entities by name or ID
        
        Args:
            query: Search query
            limit: Maximum number of results
        
        Returns:
            List of tuples containing (entity_id, entity_name)
        """
        results = []
        
        # Search by ID
        if query.startswith("CHEBI:"):
            if query in self.ontology.nodes:
                name = self.id_to_name.get(query, query)
                results.append((query, name))
        
        # Search by name
        query_lower = query.lower()
        for entity_id, name in self.id_to_name.items():
            if query_lower in name.lower():
                results.append((entity_id, name))
                if len(results) >= limit:
                    break
        
        return results
    
    def get_entity_by_id(self, entity_id: str) -> Dict:
        """
        Get entity data by ID
        
        Args:
            entity_id: ChEBI ID
        
        Returns:
            Dictionary with entity properties
        """
        if entity_id in self.ontology.nodes:
            return self.ontology.nodes[entity_id]
        return None
    
    def get_ancestors(self, concept: str, relation_type: str = "is_a") -> List[str]:
        """
        Get all ancestors of a concept following a specific relationship type
        
        Args:
            concept: Concept ID or name
            relation_type: Relation type to follow
        
        Returns:
            List of ancestor concept names
        """
        # Convert name to ID if possible
        if concept in self.name_to_id:
            concept = self.name_to_id[concept]
        
        ancestors = []
        visited = set()
        
        # Define recursive function to traverse ancestors
        def get_ancestors_recursive(node):
            if node in visited:
                return
            
            visited.add(node)
            
            for source, target, data in self.ontology.out_edges(node, data=True):
                if data.get('relation') == relation_type:
                    parent_name = self.id_to_name.get(target, target)
                    ancestors.append(parent_name)
                    get_ancestors_recursive(target)
        
        get_ancestors_recursive(concept)
        return ancestors
    
    def export_subgraph(self, concept: str, depth: int = 2, relation_types: List[str] = None, output_file: str = "subgraph.graphml"):
        """
        Export a subgraph around a concept to a GraphML file
        
        Args:
            concept: Center concept
            depth: Neighborhood depth
            relation_types: Optional list of relation types to include
            output_file: Output GraphML file path
        """
        # Convert name to ID if possible
        if concept in self.name_to_id:
            concept = self.name_to_id[concept]
        
        # Create subgraph
        subgraph = nx.MultiDiGraph()
        
        # Add center node
        if concept in self.ontology.nodes:
            node_data = self.ontology.nodes[concept]
            name = self.id_to_name.get(concept, concept)
            
            # Create a copy of node data without name attribute to avoid conflicts
            node_data_copy = {k: v for k, v in node_data.items() if k != 'name'}
            # Use a different attribute for the name
            subgraph.add_node(concept, entity_name=name, **node_data_copy)
            
            # BFS to get neighborhood
            visited = {concept}
            current_level = {concept}
            
            for _ in range(depth):
                next_level = set()
                
                for node in current_level:
                    # Process outgoing edges
                    for _, neighbor, data in self.ontology.out_edges(node, data=True):
                        rel = data.get('relation', '')
                        if relation_types is None or rel in relation_types:
                            if neighbor not in visited:
                                neighbor_data = self.ontology.nodes[neighbor]
                                neighbor_name = self.id_to_name.get(neighbor, neighbor)
                                
                                # Create a copy of node data without name attribute
                                neighbor_data_copy = {k: v for k, v in neighbor_data.items() if k != 'name'}
                                subgraph.add_node(neighbor, entity_name=neighbor_name, **neighbor_data_copy)
                                
                                subgraph.add_edge(node, neighbor, **data)
                                next_level.add(neighbor)
                                visited.add(neighbor)
                            elif neighbor not in subgraph:
                                subgraph.add_edge(node, neighbor, **data)
                    
                    # Process incoming edges
                    for neighbor, _, data in self.ontology.in_edges(node, data=True):
                        rel = data.get('relation', '')
                        if relation_types is None or rel in relation_types:
                            if neighbor not in visited:
                                neighbor_data = self.ontology.nodes[neighbor]
                                neighbor_name = self.id_to_name.get(neighbor, neighbor)
                                
                                # Create a copy of node data without name attribute
                                neighbor_data_copy = {k: v for k, v in neighbor_data.items() if k != 'name'}
                                subgraph.add_node(neighbor, entity_name=neighbor_name, **neighbor_data_copy)
                                
                                subgraph.add_edge(neighbor, node, **data)
                                next_level.add(neighbor)
                                visited.add(neighbor)
                            elif neighbor not in subgraph:
                                subgraph.add_edge(neighbor, node, **data)
                
                current_level = next_level
                if not current_level:
                    break
            
            # Export to GraphML
            try:
                nx.write_graphml(subgraph, output_file)
                print(f"Subgraph exported to {output_file} with {subgraph.number_of_nodes()} nodes and {subgraph.number_of_edges()} edges")
            except Exception as e:
                print(f"Error exporting subgraph: {e}")
        else:
            print(f"Concept {concept} not found in the ontology")


def test_basic_reasoner():
    """Test the basic ChEBI reasoner functionality"""
    # Create a directory for the ontology file
    os.makedirs("data", exist_ok=True)
    file_path = os.path.join("data", "chebi.obo")
    
    # Initialize the reasoner
    print("Initializing Basic ChEBI Reasoner...")
    reasoner = BasicChEBIReasoner(load_from_file=True, file_path=file_path)
    
    # Test search
    print("\nTesting entity search...")
    search_term = "dopamine"
    results = reasoner.search_entities(search_term)
    
    if results:
        print(f"Found {len(results)} results for '{search_term}':")
        for i, (entity_id, entity_name) in enumerate(results[:5]):
            print(f"  {i+1}. {entity_name} ({entity_id})")
        
        # Get first result
        entity_id, entity_name = results[0]
        
        # Test ancestors
        print(f"\nTesting ancestors for {entity_name}...")
        ancestors = reasoner.get_ancestors(entity_id)
        print(f"Found {len(ancestors)} ancestors:")
        for i, ancestor in enumerate(ancestors[:5]):
            print(f"  {i+1}. {ancestor}")
        
        # Test subgraph export
        print(f"\nTesting subgraph export for {entity_name}...")
        os.makedirs("output", exist_ok=True)
        output_file = os.path.join("output", f"{entity_name.replace(' ', '_')}_basic_subgraph.graphml")
        reasoner.export_subgraph(entity_id, depth=1, output_file=output_file)
    else:
        print(f"No results found for '{search_term}'")


if __name__ == "__main__":
    test_basic_reasoner()

Initializing Basic ChEBI Reasoner...
Loading ChEBI ontology from data/chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:01<00:00, 221425.28it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

Testing entity search...
Found 10 results for 'dopamine':
  1. dopamine quinone(1+) (CHEBI:167191)
  2. 3-O-Methyl-a-methyldopamine (CHEBI:173516)
  3. a-Methyldopamine (CHEBI:193722)
  4. oxidopamine (CHEBI:78741)
  5. dopamine 3-O-sulfate (CHEBI:37946)

Testing ancestors for dopamine quinone(1+)...
Found 23 ancestors:
  1. primary ammonium ion
  2. ammonium ion derivative
  3. nitrogen molecular entity
  4. pnictogen molecular entity
  5. p-block molecular entity

Testing subgraph export for dopamine quinone(1+)...


TypeError: networkx.classes.digraph.DiGraph.add_node() got multiple values for keyword argument 'entity_name'

In [13]:
import json
from typing import Dict, List, Tuple, Set, Optional, Union


class ChEBILLMReasoningInterface:
    """
    Interface between ChEBI ontology reasoning and LLM integration.
    Generates structured reasoning steps and contextual knowledge for chemical queries.
    """
    
    def __init__(self, load_from_file=True, file_path="chebi.obo"):
        """Initialize with a ChEBI reasoner instance"""
        self.reasoner = ChEBIOntologyReasoner(load_from_file, file_path)
        self.context_depth = 2  # Default depth for context gathering
    
    def set_context_depth(self, depth: int):
        """Set the depth of context to provide with responses"""
        self.context_depth = max(1, min(depth, 5))  # Limit between 1-5
    
    def generate_structured_reasoning(self, question: str) -> Dict:
        """
        Generate structured reasoning for a chemical question
        
        Args:
            question: A natural language question about chemical entities
        
        Returns:
            Dictionary with structured reasoning steps and supporting context
        """
        # Extract entities from question
        entities = self._extract_entities(question)
        
        # Generate reasoning path
        reasoning = self.reasoner.reason(question)
        
        # Enrich the reasoning with additional context
        enriched_reasoning = self._enrich_reasoning(reasoning, entities)
        
        # Format for LLM consumption
        result = {
            "question": question,
            "entities_identified": entities,
            "reasoning_steps": reasoning,
            "enriched_reasoning": enriched_reasoning,
            "context": self._gather_context(entities),
            "confidence": self._estimate_confidence(reasoning, entities)
        }
        
        return result
    
    def answer_with_reasoning(self, question: str) -> Dict:
        """
        Generate an answer with supporting reasoning for an LLM
        
        Args:
            question: A natural language question about chemical entities
        
        Returns:
            Dictionary with answer, reasoning, and confidence
        """
        # Generate structured reasoning
        reasoning_data = self.generate_structured_reasoning(question)
        
        # Derive answer from reasoning
        answer = self._derive_answer(reasoning_data)
        
        result = {
            "question": question,
            "answer": answer,
            "reasoning": reasoning_data["enriched_reasoning"],
            "context": reasoning_data["context"],
            "confidence": reasoning_data["confidence"]
        }
        
        return result
    
    def _extract_entities(self, question: str) -> List[Dict]:
        """Extract chemical entities from a question with their context"""
        # Use reasoner's entity extraction
        entity_names = self.reasoner._extract_entities_from_question(question)
        
        entities = []
        for name in entity_names:
            # Search for the entity
            results = self.reasoner.search_entities(name)
            if results:
                entity_id, entity_name = results[0]
                
                # Get entity data
                entity_data = self.reasoner.get_entity_by_id(entity_id)
                if entity_data:
                    entities.append({
                        "id": entity_id,
                        "name": entity_name,
                        "original_mention": name,
                        "properties": {k: v for k, v in entity_data.items() 
                                      if not k.startswith('_') and k != 'name'}
                    })
        
        return entities
    
    def _enrich_reasoning(self, reasoning: List[str], entities: List[Dict]) -> List[Dict]:
        """
        Enrich reasoning steps with structured data and explanations
        
        Args:
            reasoning: List of reasoning steps
            entities: List of entities involved
        
        Returns:
            List of enriched reasoning steps with explanations
        """
        enriched = []
        
        for step in reasoning:
            step_data = {
                "step": step,
                "type": self._identify_step_type(step),
                "explanation": self._generate_explanation(step, entities),
                "references": self._find_references(step)
            }
            enriched.append(step_data)
        
        return enriched
    
    def _identify_step_type(self, step: str) -> str:
        """Identify the type of reasoning step"""
        step_lower = step.lower()
        
        if "question:" in step_lower:
            return "question"
        elif "identified entities:" in step_lower:
            return "entity_identification"
        elif "classification" in step_lower:
            return "classification"
        elif "relationship" in step_lower:
            return "relationship_analysis"
        elif "property" in step_lower:
            return "property_analysis"
        elif "pathway" in step_lower or "reaction" in step_lower:
            return "pathway_reaction"
        elif "applied rule:" in step_lower:
            return "rule_application"
        elif "conclusion" in step_lower:
            return "conclusion"
        else:
            return "general"
    
    def _generate_explanation(self, step: str, entities: List[Dict]) -> str:
        """Generate an explanation for a reasoning step"""
        step_lower = step.lower()
        step_type = self._identify_step_type(step)
        
        if step_type == "entity_identification":
            return "Identifying chemical entities mentioned in the question is the first step in chemical reasoning."
        
        elif step_type == "classification":
            return "Understanding the classification hierarchy helps determine the properties and behaviors of chemical entities."
        
        elif step_type == "relationship_analysis":
            return "Analyzing relationships between entities reveals how they interact or relate to each other in chemical contexts."
        
        elif step_type == "property_analysis":
            return "Chemical properties determine how entities behave and interact in different environments."
        
        elif step_type == "pathway_reaction":
            return "Chemical pathways and reactions show how entities transform and interact in biological systems."
        
        elif step_type == "rule_application":
            rule_description = step.replace("Applied rule:", "").strip()
            return f"This is a chemical principle being applied: {rule_description}"
        
        elif step_type == "conclusion":
            return "The conclusion summarizes the key findings from the analysis of the chemical entities and their relationships."
        
        return "This step provides additional information relevant to answering the question."
    
    def _find_references(self, step: str) -> List[Dict]:
        """Find references to support a reasoning step"""
        # This would connect to reference databases or literature
        # Simplified implementation for now
        references = []
        
        # Extract entity mentions from the step
        entity_mentions = self.reasoner._extract_entities_from_question(step)
        
        for entity in entity_mentions:
            # Find the entity in ChEBI
            results = self.reasoner.search_entities(entity)
            if results:
                entity_id, entity_name = results[0]
                references.append({
                    "type": "entity_reference",
                    "id": entity_id,
                    "name": entity_name,
                    "source": "ChEBI"
                })
        
        return references
    
    def _gather_context(self, entities: List[Dict]) -> Dict:
        """
        Gather contextual information about entities to support reasoning
        
        Args:
            entities: List of entities involved in the question
        
        Returns:
            Dictionary with contextual information
        """
        context = {
            "entities": {},
            "relationships": [],
            "common_properties": []
        }
        
        # For each entity, gather context
        property_sets = []
        for entity in entities:
            entity_id = entity["id"]
            entity_name = entity["name"]
            
            # Get ancestors (classifications)
            ancestors = self.reasoner.get_ancestors(entity_id)
            
            # Get related entities
            related = self.reasoner.get_related_concepts(entity_id)
            related_formatted = [
                {"source": src, "target": tgt, "relation": rel}
                for src, tgt, rel in related[:10]  # Limit to first 10
            ]
            
            # Get entity data
            entity_data = self.reasoner.get_entity_by_id(entity_id)
            
            # Store context for this entity
            context["entities"][entity_name] = {
                "id": entity_id,
                "classifications": ancestors[:5],  # Top 5 classifications
                "related_entities": related_formatted,
                "properties": {k: v for k, v in entity_data.items() 
                              if not k.startswith('_') and k != 'name'}
            }
            
            # Track properties for finding common ones
            property_sets.append(set(context["entities"][entity_name]["properties"].keys()))
        
        # Find relationships between the entities
        if len(entities) >= 2:
            for i in range(len(entities)):
                for j in range(i+1, len(entities)):
                    entity1 = entities[i]
                    entity2 = entities[j]
                    
                    entity1_id = entity1["id"]
                    entity2_id = entity2["id"]
                    
                    # Try to find direct relationships
                    related = []
                    for _, _, data in self.reasoner.ontology.get_edge_data(entity1_id, entity2_id, default=[]).items():
                        relation = data.get('relation', 'related_to')
                        related.append({
                            "source": entity1["name"],
                            "target": entity2["name"],
                            "relation": relation
                        })
                    
                    for _, _, data in self.reasoner.ontology.get_edge_data(entity2_id, entity1_id, default=[]).items():
                        relation = data.get('relation', 'related_to')
                        related.append({
                            "source": entity2["name"],
                            "target": entity1["name"],
                            "relation": relation
                        })
                    
                    context["relationships"].extend(related)
        
        # Find common properties across entities
        if len(property_sets) >= 2:
            common_properties = set.intersection(*property_sets)
            context["common_properties"] = list(common_properties)
        
        return context
    
    def _derive_answer(self, reasoning_data: Dict) -> Dict:
        """
        Derive an answer from reasoning data
        
        Args:
            reasoning_data: Structured reasoning data
        
        Returns:
            Dictionary with derived answer and explanation
        """
        # Extract information from reasoning
        question = reasoning_data["question"]
        entities = reasoning_data["entities_identified"]
        steps = reasoning_data["reasoning_steps"]
        
        # Find conclusion step if any
        conclusion = None
        for step in steps:
            if "conclusion" in step.lower():
                conclusion = step
                break
        
        # Identify question type
        if "what is" in question.lower():
            question_type = "definition"
        elif "how does" in question.lower() or "how do" in question.lower():
            question_type = "mechanism"
        elif "relationship" in question.lower() or "related" in question.lower():
            question_type = "relationship"
        elif "property" in question.lower() or "properties" in question.lower():
            question_type = "property"
        elif "classify" in question.lower() or "classification" in question.lower():
            question_type = "classification"
        else:
            question_type = "general"
        
        # Generate an answer based on question type and reasoning
        if question_type == "definition" and entities:
            entity = entities[0]["name"]
            ancestors = self.reasoner.get_ancestors(entities[0]["id"])[:3]
            answer = {
                "short": f"{entity} is a {' and '.join(ancestors)}.",
                "detailed": f"{entity} is classified as {', '.join(ancestors)}. " + 
                           (conclusion or "It has specific properties and relationships in the ChEBI ontology.")
            }
        
        elif question_type == "classification" and entities:
            entity = entities[0]["name"]
            ancestors = self.reasoner.get_ancestors(entities[0]["id"])[:5]
            answer = {
                "short": f"{entity} is classified as {', '.join(ancestors[:2])}.",
                "detailed": f"{entity} belongs to the following classifications: {', '.join(ancestors)}. " +
                           (conclusion or "")
            }
        
        elif question_type == "relationship" and len(entities) >= 2:
            entity1 = entities[0]["name"]
            entity2 = entities[1]["name"]
            
            # Find relationships in context
            relationships = reasoning_data["context"]["relationships"]
            if relationships:
                rel_str = ", ".join([f"{r['source']} {r['relation']} {r['target']}" for r in relationships[:2]])
                answer = {
                    "short": f"{entity1} and {entity2} are related: {rel_str}.",
                    "detailed": "The relationship between these entities includes: " + 
                               ", ".join([f"{r['source']} {r['relation']} {r['target']}" for r in relationships]) +
                               (f" {conclusion}" if conclusion else "")
                }
            else:
                # Find common classifications
                entity1_ancestors = set(self.reasoner.get_ancestors(entities[0]["id"]))
                entity2_ancestors = set(self.reasoner.get_ancestors(entities[1]["id"]))
                common = entity1_ancestors.intersection(entity2_ancestors)
                
                if common:
                    answer = {
                        "short": f"{entity1} and {entity2} share classifications: {', '.join(list(common)[:2])}.",
                        "detailed": f"While no direct relationship is recorded, {entity1} and {entity2} " +
                                  f"share these classifications: {', '.join(list(common)[:5])}." +
                                  (f" {conclusion}" if conclusion else "")
                    }
                else:
                    answer = {
                        "short": f"No direct relationship found between {entity1} and {entity2}.",
                        "detailed": f"The ChEBI ontology does not record a direct relationship between {entity1} and {entity2}."
                    }
        
        else:
            # General answer based on available reasoning
            if conclusion:
                answer = {
                    "short": conclusion.replace("Conclusion: ", ""),
                    "detailed": conclusion
                }
            else:
                # Use the last non-question step as a fallback
                for step in reversed(steps):
                    if "question:" not in step.lower():
                        answer = {
                            "short": step,
                            "detailed": step
                        }
                        break
                else:
                    answer = {
                        "short": "Unable to derive an answer from the available information.",
                        "detailed": "The ChEBI ontology does not contain sufficient information to answer this question."
                    }
        
        return answer
    
    def _estimate_confidence(self, reasoning: List[str], entities: List[Dict]) -> float:
        """
        Estimate the confidence in the reasoning and answer
        
        Args:
            reasoning: List of reasoning steps
            entities: List of entities identified
        
        Returns:
            Confidence score between 0.0 and 1.0
        """
        # Base confidence on several factors
        confidence = 0.5  # Start with neutral confidence
        
        # Factor 1: Were entities found?
        if not entities:
            return 0.2  # Very low confidence if no entities were found
        
        confidence += min(0.2, 0.05 * len(entities))  # Up to 0.2 boost for found entities
        
        # Factor 2: Entity coverage in question
        question_words = set(reasoning[0].lower().replace("question: ", "").split())
        entity_words = set()
        for entity in entities:
            entity_words.update(entity["name"].lower().split())
            entity_words.update(entity["original_mention"].lower().split())
        
        coverage = len(entity_words.intersection(question_words)) / max(1, len(question_words))
        confidence += 0.1 * coverage  # Up to 0.1 boost for coverage
        
        # Factor 3: Reasoning steps depth
        if len(reasoning) <= 2:  # Just question and one step
            confidence -= 0.1
        elif len(reasoning) >= 5:  # Detailed reasoning
            confidence += 0.1
        
        # Factor 4: Conclusion presence
        has_conclusion = any("conclusion" in step.lower() for step in reasoning)
        if has_conclusion:
            confidence += 0.1
        
        # Ensure confidence is between 0.0 and 1.0
        return max(0.0, min(1.0, confidence))
    
    def to_json(self, data: Dict) -> str:
        """Convert a reasoning or answer structure to JSON for LLM consumption"""
        return json.dumps(data, indent=2)
    
    def answer_batch(self, questions: List[str]) -> List[Dict]:
        """
        Process a batch of questions with reasoning
        
        Args:
            questions: List of questions to answer
        
        Returns:
            List of answer dictionaries with reasoning
        """
        return [self.answer_with_reasoning(q) for q in questions]


def demonstrate_llm_integration():
    """Demonstrate the LLM reasoning interface"""
    print("Initializing ChEBI LLM Reasoning Interface...")
    interface = ChEBILLMReasoningInterface(load_from_file=True)
    
    # Example questions
    questions = [
        "What is dopamine and how is it classified?",
        "What is the relationship between glucose and ATP?",
        "What properties does acetylcholine have?",
        "How are serotonin and dopamine related?",
        "What pathway involves glucose metabolism?"
    ]
    
    for question in questions:
        print(f"\n=== Answering: {question} ===")
        
        # Get answer with reasoning
        result = interface.answer_with_reasoning(question)
        
        # Display results
        print(f"Answer: {result['answer']['short']}")
        print(f"Confidence: {result['confidence']:.2f}")
        print("\nDetailed answer:")
        print(result['answer']['detailed'])
        
        print("\nReasoning steps:")
        for i, step in enumerate(result['reasoning']):
            print(f"  {i+1}. {step['step']}")
            print(f"     Type: {step['type']}")
            print(f"     Explanation: {step['explanation']}")
        
        print("\nKey context:")
        for entity_name, entity_data in result['context']['entities'].items():
            print(f"  - {entity_name}: {', '.join(entity_data['classifications'][:3])}")
        
        if result['context']['relationships']:
            print("  Relationships:")
            for rel in result['context']['relationships'][:3]:
                print(f"    - {rel['source']} {rel['relation']} {rel['target']}")
        
        print("\n" + "="*50)
    
    # Example of JSON output for LLM
    json_output = interface.to_json(result)
    print("\nJSON output for LLM consumption (example):")
    print(json_output[:500] + "... (truncated)")


if __name__ == "__main__":
    demonstrate_llm_integration()

Initializing ChEBI LLM Reasoning Interface...
Loading ChEBI ontology from chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:01<00:00, 218043.50it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

=== Answering: What is dopamine and how is it classified? ===


AttributeError: 'list' object has no attribute 'items'

In [15]:
import os
import json
import requests
from typing import Dict, List, Optional, Any


class ChEBILLMIntegration:
    """
    Integration between the ChEBI reasoner and an LLM API
    to provide accurate chemical reasoning capabilities.
    """
    
    def __init__(self, llm_api_endpoint: str = None, llm_api_key: str = None):
        """
        Initialize the integration with ChEBI reasoner and LLM
        
        Args:
            llm_api_endpoint: LLM API endpoint URL 
            llm_api_key: API key for the LLM service
        """
        # Initialize the reasoning interface
        self.reasoning_interface = ChEBILLMReasoningInterface(load_from_file=True)
        
        # LLM API configuration
        self.llm_api_endpoint = llm_api_endpoint or os.environ.get("LLM_API_ENDPOINT")
        self.llm_api_key = llm_api_key or os.environ.get("LLM_API_KEY")
        
        # Default prompt templates
        self.system_prompt_template = """
You are a chemical reasoning assistant with access to the ChEBI (Chemical Entities of Biological Interest) ontology.
You will receive a question along with structured reasoning derived from the ChEBI ontology.
Your task is to provide a clear, scientifically accurate answer that follows the reasoning provided.

Use the following approach:
1. Ensure your answer is consistent with the reasoning steps provided
2. Incorporate the contextual information about chemical entities
3. Express confidence levels appropriately based on the data provided
4. Cite relationships and classifications from ChEBI when relevant
5. Maintain scientific accuracy while being accessible to the user

The reasoning provided is based on ChEBI, a highly reliable scientific ontology for biochemical entities.
        """
        
        self.user_prompt_template = """
QUESTION: {question}

CHEMICAL REASONING:
{reasoning_summary}

ENTITY CONTEXT:
{entity_context}

Based on the chemical reasoning provided, please answer the question accurately and clearly.
Explain your reasoning process in a way that's scientifically sound but understandable.
If the provided reasoning indicates uncertainty, acknowledge the limitations in your answer.
"""
    
    def set_llm_api(self, endpoint: str, api_key: str):
        """Set the LLM API endpoint and key"""
        self.llm_api_endpoint = endpoint
        self.llm_api_key = api_key
    
    def set_prompt_templates(self, system_prompt: str = None, user_prompt: str = None):
        """Set custom prompt templates for the LLM"""
        if system_prompt:
            self.system_prompt_template = system_prompt
        if user_prompt:
            self.user_prompt_template = user_prompt
    
    def process_chemical_question(self, question: str, use_llm: bool = True) -> Dict:
        """
        Process a chemical question using ChEBI reasoning and optionally an LLM
        
        Args:
            question: The chemical question to process
            use_llm: Whether to use the LLM to generate the final answer
        
        Returns:
            Dictionary with the processed answer and reasoning
        """
        # Generate reasoning using the ChEBI ontology
        reasoning_result = self.reasoning_interface.answer_with_reasoning(question)
        
        if not use_llm or not self.llm_api_endpoint:
            # Return only the reasoning interface result if LLM is not used
            return {
                "question": question,
                "answer": reasoning_result["answer"]["detailed"],
                "confidence": reasoning_result["confidence"],
                "reasoning": [step["step"] for step in reasoning_result["reasoning"]],
                "source": "ChEBI Ontology"
            }
        
        # Prepare LLM prompt with reasoning
        llm_response = self._query_llm(reasoning_result)
        
        # Combine results
        combined_result = {
            "question": question,
            "answer": llm_response["content"] if llm_response else reasoning_result["answer"]["detailed"],
            "confidence": reasoning_result["confidence"],
            "reasoning": [step["step"] for step in reasoning_result["reasoning"]],
            "context": self._format_context_summary(reasoning_result["context"]),
            "source": "ChEBI Ontology + LLM" if llm_response else "ChEBI Ontology"
        }
        
        return combined_result
    
    def _query_llm(self, reasoning_result: Dict) -> Optional[Dict]:
        """
        Query the LLM API with the reasoning data
        
        Args:
            reasoning_result: The reasoning data from the ChEBI interface
        
        Returns:
            LLM response or None if the request failed
        """
        if not self.llm_api_endpoint or not self.llm_api_key:
            print("LLM API not configured. Set endpoint and API key first.")
            return None
        
        try:
            # Format the reasoning for the LLM
            reasoning_summary = self._format_reasoning_summary(reasoning_result["reasoning"])
            entity_context = self._format_entity_context(reasoning_result["context"])
            
            # Format the prompt
            user_prompt = self.user_prompt_template.format(
                question=reasoning_result["question"],
                reasoning_summary=reasoning_summary,
                entity_context=entity_context
            )
            
            # Prepare the API request
            headers = {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.llm_api_key}"
            }
            
            payload = {
                "model": "gpt-4",  # Or another model identifier
                "messages": [
                    {"role": "system", "content": self.system_prompt_template},
                    {"role": "user", "content": user_prompt}
                ],
                "temperature": 0.2,  # Low temperature for more deterministic outputs
                "max_tokens": 1000
            }
            
            # Make the API request
            response = requests.post(
                self.llm_api_endpoint,
                headers=headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                if "choices" in result and len(result["choices"]) > 0:
                    return result["choices"][0]["message"]
            
            print(f"LLM API request failed with status code: {response.status_code}")
            print(f"Response: {response.text}")
            return None
        
        except Exception as e:
            print(f"Error querying LLM API: {e}")
            return None
    
    def _format_reasoning_summary(self, reasoning: List[Dict]) -> str:
        """Format reasoning steps for LLM consumption"""
        summary = ""
        for i, step in enumerate(reasoning):
            summary += f"{i+1}. {step['step']}\n"
            if step.get('explanation'):
                summary += f"   Explanation: {step['explanation']}\n"
        
        return summary
    
    def _format_entity_context(self, context: Dict) -> str:
        """Format entity context for LLM consumption"""
        formatted = "Known entities and their properties:\n"
        
        for entity_name, entity_data in context["entities"].items():
            formatted += f"\n- {entity_name}:\n"
            formatted += f"  Classifications: {', '.join(entity_data['classifications'][:3])}\n"
            
            if entity_data["properties"]:
                props = []
                for k, v in entity_data["properties"].items():
                    if not k.startswith('_'):
                        props.append(f"{k}: {v}")
                if props:
                    formatted += f"  Properties: {'; '.join(props[:3])}\n"
        
        if context["relationships"]:
            formatted += "\nRelationships between entities:\n"
            for rel in context["relationships"][:5]:
                formatted += f"- {rel['source']} {rel['relation']} {rel['target']}\n"
        
        return formatted
    
    def _format_context_summary(self, context: Dict) -> Dict:
        """Format context for the response"""
        return {
            "entities": list(context["entities"].keys()),
            "key_classifications": {
                name: data["classifications"][:3] 
                for name, data in context["entities"].items()
            },
            "relationships": [
                f"{rel['source']} {rel['relation']} {rel['target']}"
                for rel in context["relationships"][:3]
            ]
        }
    
    def save_response(self, result: Dict, filename: str = None):
        """Save a response to a JSON file"""
        if filename is None:
            # Generate a filename based on the question
            question_words = result["question"].lower().split()[:5]
            filename = f"{'_'.join(question_words)}.json"
        
        with open(filename, 'w') as f:
            json.dump(result, f, indent=2)
        
        print(f"Response saved to {filename}")


def demonstrate_llm_integration():
    """Demonstrate the LLM integration"""
    print("Initializing ChEBI LLM Integration...")
    
    # For the demonstration, we'll use a mock LLM response
    integration = ChEBILLMIntegration()
    
    # Example question
    question = "What is the relationship between dopamine and serotonin in neurotransmission?"
    
    print(f"\n=== Processing question: {question} ===\n")
    
    # First show reasoning without LLM
    print("Getting reasoning from ChEBI ontology...")
    result_without_llm = integration.process_chemical_question(question, use_llm=False)
    
    print("\nChEBI Reasoning steps:")
    for step in result_without_llm["reasoning"]:
        print(f"- {step}")
    
    print(f"\nConfidence: {result_without_llm['confidence']:.2f}")
    print("\nAnswer based on ChEBI only:")
    print(result_without_llm["answer"])
    
    print("\n" + "="*50)
    print("\nIn a production environment, the system would now:")
    print("1. Send this reasoning to an LLM API (like GPT-4)")
    print("2. The LLM would generate a coherent, scientifically accurate answer")
    print("3. The answer would incorporate the ChEBI reasoning steps")
    print("4. The final response would combine ontological accuracy with natural language fluency")
    
    # Mock what an LLM-enhanced answer might look like
    mock_llm_answer = """
Based on the ChEBI ontology analysis, dopamine and serotonin are both classified as neurotransmitters, specifically monoamine neurotransmitters. They share several functional similarities but have distinct roles in neurotransmission.

Dopamine is primarily involved in reward pathways, motor control, and executive functions. It's synthesized from tyrosine and functions through dopaminergic receptors.

Serotonin (5-hydroxytryptamine) is involved in regulating mood, appetite, sleep, and cognitive functions. It's synthesized from tryptophan and acts through serotonergic receptors.

While they don't directly interact in a chemical sense, these neurotransmitters often work in balance within the brain. For example, imbalances between dopamine and serotonin systems have been implicated in various neurological and psychiatric conditions. Several medications target both systems either directly or indirectly.

The ChEBI ontology confirms that both compounds share classifications as neurotransmitters and endogenous metabolites, supporting their related but distinct biological roles.
"""
    
    print("\nExample of what an LLM-enhanced answer might look like:")
    print(mock_llm_answer)


if __name__ == "__main__":
    demonstrate_llm_integration()

Initializing ChEBI LLM Integration...
Loading ChEBI ontology from chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:01<00:00, 302533.58it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

=== Processing question: What is the relationship between dopamine and serotonin in neurotransmission? ===

Getting reasoning from ChEBI ontology...


AttributeError: 'list' object has no attribute 'items'

In [17]:
import os
import json
import requests
from typing import Dict, List, Optional, Any



class ChEBILLMIntegration:
    """
    Integration between the ChEBI reasoner and an LLM API
    to provide accurate chemical reasoning capabilities.
    """
    
    def __init__(self, llm_api_endpoint: str = None, llm_api_key: str = None):
        """
        Initialize the integration with ChEBI reasoner and LLM
        
        Args:
            llm_api_endpoint: LLM API endpoint URL 
            llm_api_key: API key for the LLM service
        """
        # Initialize the reasoning interface
        self.reasoning_interface = ChEBILLMReasoningInterface(load_from_file=True)
        
        # LLM API configuration
        self.llm_api_endpoint = llm_api_endpoint or os.environ.get("LLM_API_ENDPOINT")
        self.llm_api_key = llm_api_key or os.environ.get("LLM_API_KEY")
        
        # Default prompt templates
        self.system_prompt_template = """
You are a chemical reasoning assistant with access to the ChEBI (Chemical Entities of Biological Interest) ontology.
You will receive a question along with structured reasoning derived from the ChEBI ontology.
Your task is to provide a clear, scientifically accurate answer that follows the reasoning provided.

Use the following approach:
1. Ensure your answer is consistent with the reasoning steps provided
2. Incorporate the contextual information about chemical entities
3. Express confidence levels appropriately based on the data provided
4. Cite relationships and classifications from ChEBI when relevant
5. Maintain scientific accuracy while being accessible to the user

The reasoning provided is based on ChEBI, a highly reliable scientific ontology for biochemical entities.
        """
        
        self.user_prompt_template = """
QUESTION: {question}

CHEMICAL REASONING:
{reasoning_summary}

ENTITY CONTEXT:
{entity_context}

Based on the chemical reasoning provided, please answer the question accurately and clearly.
Explain your reasoning process in a way that's scientifically sound but understandable.
If the provided reasoning indicates uncertainty, acknowledge the limitations in your answer.
"""
    
    def set_llm_api(self, endpoint: str, api_key: str):
        """Set the LLM API endpoint and key"""
        self.llm_api_endpoint = endpoint
        self.llm_api_key = api_key
    
    def set_prompt_templates(self, system_prompt: str = None, user_prompt: str = None):
        """Set custom prompt templates for the LLM"""
        if system_prompt:
            self.system_prompt_template = system_prompt
        if user_prompt:
            self.user_prompt_template = user_prompt
    
    def process_chemical_question(self, question: str, use_llm: bool = True) -> Dict:
        """
        Process a chemical question using ChEBI reasoning and optionally an LLM
        
        Args:
            question: The chemical question to process
            use_llm: Whether to use the LLM to generate the final answer
        
        Returns:
            Dictionary with the processed answer and reasoning
        """
        # Generate reasoning using the ChEBI ontology
        reasoning_result = self.reasoning_interface.answer_with_reasoning(question)
        
        if not use_llm or not self.llm_api_endpoint:
            # Return only the reasoning interface result if LLM is not used
            return {
                "question": question,
                "answer": reasoning_result["answer"]["detailed"],
                "confidence": reasoning_result["confidence"],
                "reasoning": [step["step"] for step in reasoning_result["reasoning"]],
                "source": "ChEBI Ontology"
            }
        
        # Prepare LLM prompt with reasoning
        llm_response = self._query_llm(reasoning_result)
        
        # Combine results
        combined_result = {
            "question": question,
            "answer": llm_response["content"] if llm_response else reasoning_result["answer"]["detailed"],
            "confidence": reasoning_result["confidence"],
            "reasoning": [step["step"] for step in reasoning_result["reasoning"]],
            "context": self._format_context_summary(reasoning_result["context"]),
            "source": "ChEBI Ontology + LLM" if llm_response else "ChEBI Ontology"
        }
        
        return combined_result
    
    def _query_llm(self, reasoning_result: Dict) -> Optional[Dict]:
        """
        Query the LLM API with the reasoning data
        
        Args:
            reasoning_result: The reasoning data from the ChEBI interface
        
        Returns:
            LLM response or None if the request failed
        """
        if not self.llm_api_endpoint or not self.llm_api_key:
            print("LLM API not configured. Set endpoint and API key first.")
            return None
        
        try:
            # Format the reasoning for the LLM
            reasoning_summary = self._format_reasoning_summary(reasoning_result["reasoning"])
            entity_context = self._format_entity_context(reasoning_result["context"])
            
            # Format the prompt
            user_prompt = self.user_prompt_template.format(
                question=reasoning_result["question"],
                reasoning_summary=reasoning_summary,
                entity_context=entity_context
            )
            
            # Prepare the API request
            headers = {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.llm_api_key}"
            }
            
            payload = {
                "model": "gpt-4",  # Or another model identifier
                "messages": [
                    {"role": "system", "content": self.system_prompt_template},
                    {"role": "user", "content": user_prompt}
                ],
                "temperature": 0.2,  # Low temperature for more deterministic outputs
                "max_tokens": 1000
            }
            
            # Make the API request
            response = requests.post(
                self.llm_api_endpoint,
                headers=headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                if "choices" in result and len(result["choices"]) > 0:
                    return result["choices"][0]["message"]
            
            print(f"LLM API request failed with status code: {response.status_code}")
            print(f"Response: {response.text}")
            return None
        
        except Exception as e:
            print(f"Error querying LLM API: {e}")
            return None
    
    def _format_reasoning_summary(self, reasoning: List[Dict]) -> str:
        """Format reasoning steps for LLM consumption"""
        summary = ""
        for i, step in enumerate(reasoning):
            summary += f"{i+1}. {step['step']}\n"
            if step.get('explanation'):
                summary += f"   Explanation: {step['explanation']}\n"
        
        return summary
    
    def _format_entity_context(self, context: Dict) -> str:
        """Format entity context for LLM consumption"""
        formatted = "Known entities and their properties:\n"
        
        for entity_name, entity_data in context["entities"].items():
            formatted += f"\n- {entity_name}:\n"
            formatted += f"  Classifications: {', '.join(entity_data['classifications'][:3])}\n"
            
            if entity_data["properties"]:
                props = []
                for k, v in entity_data["properties"].items():
                    if not k.startswith('_'):
                        props.append(f"{k}: {v}")
                if props:
                    formatted += f"  Properties: {'; '.join(props[:3])}\n"
        
        if context["relationships"]:
            formatted += "\nRelationships between entities:\n"
            for rel in context["relationships"][:5]:
                formatted += f"- {rel['source']} {rel['relation']} {rel['target']}\n"
        
        return formatted
    
    def _format_context_summary(self, context: Dict) -> Dict:
        """Format context for the response"""
        return {
            "entities": list(context["entities"].keys()),
            "key_classifications": {
                name: data["classifications"][:3] 
                for name, data in context["entities"].items()
            },
            "relationships": [
                f"{rel['source']} {rel['relation']} {rel['target']}"
                for rel in context["relationships"][:3]
            ]
        }
    
    def save_response(self, result: Dict, filename: str = None):
        """Save a response to a JSON file"""
        if filename is None:
            # Generate a filename based on the question
            question_words = result["question"].lower().split()[:5]
            filename = f"{'_'.join(question_words)}.json"
        
        with open(filename, 'w') as f:
            json.dump(result, f, indent=2)
        
        print(f"Response saved to {filename}")


def demonstrate_llm_integration():
    """Demonstrate the LLM integration"""
    print("Initializing ChEBI LLM Integration...")
    
    # For the demonstration, we'll use a mock LLM response
    integration = ChEBILLMIntegration()
    
    # Example question
    question = "What is the relationship between dopamine and serotonin in neurotransmission?"
    
    print(f"\n=== Processing question: {question} ===\n")
    
    # First show reasoning without LLM
    print("Getting reasoning from ChEBI ontology...")
    result_without_llm = integration.process_chemical_question(question, use_llm=False)
    
    print("\nChEBI Reasoning steps:")
    for step in result_without_llm["reasoning"]:
        print(f"- {step}")
    
    print(f"\nConfidence: {result_without_llm['confidence']:.2f}")
    print("\nAnswer based on ChEBI only:")
    print(result_without_llm["answer"])
    
    print("\n" + "="*50)
    print("\nIn a production environment, the system would now:")
    print("1. Send this reasoning to an LLM API (like GPT-4)")
    print("2. The LLM would generate a coherent, scientifically accurate answer")
    print("3. The answer would incorporate the ChEBI reasoning steps")
    print("4. The final response would combine ontological accuracy with natural language fluency")
    
    # Mock what an LLM-enhanced answer might look like
    mock_llm_answer = """
Based on the ChEBI ontology analysis, dopamine and serotonin are both classified as neurotransmitters, specifically monoamine neurotransmitters. They share several functional similarities but have distinct roles in neurotransmission.

Dopamine is primarily involved in reward pathways, motor control, and executive functions. It's synthesized from tyrosine and functions through dopaminergic receptors.

Serotonin (5-hydroxytryptamine) is involved in regulating mood, appetite, sleep, and cognitive functions. It's synthesized from tryptophan and acts through serotonergic receptors.

While they don't directly interact in a chemical sense, these neurotransmitters often work in balance within the brain. For example, imbalances between dopamine and serotonin systems have been implicated in various neurological and psychiatric conditions. Several medications target both systems either directly or indirectly.

The ChEBI ontology confirms that both compounds share classifications as neurotransmitters and endogenous metabolites, supporting their related but distinct biological roles.
"""
    
    print("\nExample of what an LLM-enhanced answer might look like:")
    print(mock_llm_answer)


if __name__ == "__main__":
    demonstrate_llm_integration()

Initializing ChEBI LLM Integration...
Loading ChEBI ontology from chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:01<00:00, 303319.65it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

=== Processing question: What is the relationship between dopamine and serotonin in neurotransmission? ===

Getting reasoning from ChEBI ontology...


AttributeError: 'list' object has no attribute 'items'

In [18]:
import json
from typing import Dict, List, Tuple, Set, Optional, Union
from chebi_reasoner import ChEBIOntologyReasoner

class ChEBILLMReasoningInterface:
    """
    Interface between ChEBI ontology reasoning and LLM integration.
    Generates structured reasoning steps and contextual knowledge for chemical queries.
    """
    
    def __init__(self, load_from_file=True, file_path="chebi.obo"):
        """Initialize with a ChEBI reasoner instance"""
        self.reasoner = ChEBIOntologyReasoner(load_from_file, file_path)
        self.context_depth = 2  # Default depth for context gathering
    
    def set_context_depth(self, depth: int):
        """Set the depth of context to provide with responses"""
        self.context_depth = max(1, min(depth, 5))  # Limit between 1-5
    
    def generate_structured_reasoning(self, question: str) -> Dict:
        """
        Generate structured reasoning for a chemical question
        
        Args:
            question: A natural language question about chemical entities
        
        Returns:
            Dictionary with structured reasoning steps and supporting context
        """
        # Extract entities from question
        entities = self._extract_entities(question)
        
        # Generate reasoning path
        reasoning = self.reasoner.reason(question)
        
        # Enrich the reasoning with additional context
        enriched_reasoning = self._enrich_reasoning(reasoning, entities)
        
        # Format for LLM consumption
        result = {
            "question": question,
            "entities_identified": entities,
            "reasoning_steps": reasoning,
            "enriched_reasoning": enriched_reasoning,
            "context": self._gather_context(entities),
            "confidence": self._estimate_confidence(reasoning, entities)
        }
        
        return result
    
    def answer_with_reasoning(self, question: str) -> Dict:
        """
        Generate an answer with supporting reasoning for an LLM
        
        Args:
            question: A natural language question about chemical entities
        
        Returns:
            Dictionary with answer, reasoning, and confidence
        """
        # Generate structured reasoning
        reasoning_data = self.generate_structured_reasoning(question)
        
        # Derive answer from reasoning
        answer = self._derive_answer(reasoning_data)
        
        result = {
            "question": question,
            "answer": answer,
            "reasoning": reasoning_data["enriched_reasoning"],
            "context": reasoning_data["context"],
            "confidence": reasoning_data["confidence"]
        }
        
        return result
    
    def _extract_entities(self, question: str) -> List[Dict]:
        """Extract chemical entities from a question with their context"""
        # Use reasoner's entity extraction
        entity_names = self.reasoner._extract_entities_from_question(question)
        
        entities = []
        for name in entity_names:
            # Search for the entity
            results = self.reasoner.search_entities(name)
            if results:
                entity_id, entity_name = results[0]
                
                # Get entity data
                entity_data = self.reasoner.get_entity_by_id(entity_id)
                if entity_data:
                    entities.append({
                        "id": entity_id,
                        "name": entity_name,
                        "original_mention": name,
                        "properties": {k: v for k, v in entity_data.items() 
                                      if not k.startswith('_') and k != 'name'}
                    })
        
        return entities
    
    def _enrich_reasoning(self, reasoning: List[str], entities: List[Dict]) -> List[Dict]:
        """
        Enrich reasoning steps with structured data and explanations
        
        Args:
            reasoning: List of reasoning steps
            entities: List of entities involved
        
        Returns:
            List of enriched reasoning steps with explanations
        """
        enriched = []
        
        for step in reasoning:
            step_data = {
                "step": step,
                "type": self._identify_step_type(step),
                "explanation": self._generate_explanation(step, entities),
                "references": self._find_references(step)
            }
            enriched.append(step_data)
        
        return enriched
    
    def _identify_step_type(self, step: str) -> str:
        """Identify the type of reasoning step"""
        step_lower = step.lower()
        
        if "question:" in step_lower:
            return "question"
        elif "identified entities:" in step_lower:
            return "entity_identification"
        elif "classification" in step_lower:
            return "classification"
        elif "relationship" in step_lower:
            return "relationship_analysis"
        elif "property" in step_lower:
            return "property_analysis"
        elif "pathway" in step_lower or "reaction" in step_lower:
            return "pathway_reaction"
        elif "applied rule:" in step_lower:
            return "rule_application"
        elif "conclusion" in step_lower:
            return "conclusion"
        else:
            return "general"
    
    def _generate_explanation(self, step: str, entities: List[Dict]) -> str:
        """Generate an explanation for a reasoning step"""
        step_lower = step.lower()
        step_type = self._identify_step_type(step)
        
        if step_type == "entity_identification":
            return "Identifying chemical entities mentioned in the question is the first step in chemical reasoning."
        
        elif step_type == "classification":
            return "Understanding the classification hierarchy helps determine the properties and behaviors of chemical entities."
        
        elif step_type == "relationship_analysis":
            return "Analyzing relationships between entities reveals how they interact or relate to each other in chemical contexts."
        
        elif step_type == "property_analysis":
            return "Chemical properties determine how entities behave and interact in different environments."
        
        elif step_type == "pathway_reaction":
            return "Chemical pathways and reactions show how entities transform and interact in biological systems."
        
        elif step_type == "rule_application":
            rule_description = step.replace("Applied rule:", "").strip()
            return f"This is a chemical principle being applied: {rule_description}"
        
        elif step_type == "conclusion":
            return "The conclusion summarizes the key findings from the analysis of the chemical entities and their relationships."
        
        return "This step provides additional information relevant to answering the question."
    
    def _find_references(self, step: str) -> List[Dict]:
        """Find references to support a reasoning step"""
        # This would connect to reference databases or literature
        # Simplified implementation for now
        references = []
        
        # Extract entity mentions from the step
        entity_mentions = self.reasoner._extract_entities_from_question(step)
        
        for entity in entity_mentions:
            # Find the entity in ChEBI
            results = self.reasoner.search_entities(entity)
            if results:
                entity_id, entity_name = results[0]
                references.append({
                    "type": "entity_reference",
                    "id": entity_id,
                    "name": entity_name,
                    "source": "ChEBI"
                })
        
        return references
    
    def _gather_context(self, entities: List[Dict]) -> Dict:
        """
        Gather contextual information about entities to support reasoning
        
        Args:
            entities: List of entities involved in the question
        
        Returns:
            Dictionary with contextual information
        """
        context = {
            "entities": {},
            "relationships": [],
            "common_properties": []
        }
        
        # For each entity, gather context
        property_sets = []
        for entity in entities:
            entity_id = entity["id"]
            entity_name = entity["name"]
            
            # Get ancestors (classifications)
            ancestors = self.reasoner.get_ancestors(entity_id)
            
            # Get related entities
            related = self.reasoner.get_related_concepts(entity_id)
            related_formatted = [
                {"source": src, "target": tgt, "relation": rel}
                for src, tgt, rel in related[:10]  # Limit to first 10
            ]
            
            # Get entity data
            entity_data = self.reasoner.get_entity_by_id(entity_id)
            
            # Store context for this entity
            context["entities"][entity_name] = {
                "id": entity_id,
                "classifications": ancestors[:5],  # Top 5 classifications
                "related_entities": related_formatted,
                "properties": {k: v for k, v in entity_data.items() 
                              if not k.startswith('_') and k != 'name'}
            }
            
            # Track properties for finding common ones
            property_sets.append(set(context["entities"][entity_name]["properties"].keys()))
        
        # Find relationships between the entities
        if len(entities) >= 2:
            for i in range(len(entities)):
                for j in range(i+1, len(entities)):
                    entity1 = entities[i]
                    entity2 = entities[j]
                    
                    entity1_id = entity1["id"]
                    entity2_id = entity2["id"]
                    
                    # Try to find direct relationships
                    related = []
                    edge_data = self.reasoner.ontology.get_edge_data(entity1_id, entity2_id)
                    if edge_data:
                        for key, data in edge_data.items():
                            relation = data.get('relation', 'related_to')
                            related.append({
                                "source": entity1["name"],
                                "target": entity2["name"],
                                "relation": relation
                            })
                    
                    edge_data = self.reasoner.ontology.get_edge_data(entity2_id, entity1_id)
                    if edge_data:
                        for key, data in edge_data.items():
                            relation = data.get('relation', 'related_to')
                            related.append({
                                "source": entity2["name"],
                                "target": entity1["name"], 
                                "relation": relation
                            })
                    context["relationships"].extend(related)
        
        # Find common properties across entities
        if len(property_sets) >= 2:
            common_properties = set.intersection(*property_sets)
            context["common_properties"] = list(common_properties)
        
        return context
    
    def _derive_answer(self, reasoning_data: Dict) -> Dict:
        """
        Derive an answer from reasoning data
        
        Args:
            reasoning_data: Structured reasoning data
        
        Returns:
            Dictionary with derived answer and explanation
        """
        # Extract information from reasoning
        question = reasoning_data["question"]
        entities = reasoning_data["entities_identified"]
        steps = reasoning_data["reasoning_steps"]
        
        # Find conclusion step if any
        conclusion = None
        for step in steps:
            if "conclusion" in step.lower():
                conclusion = step
                break
        
        # Identify question type
        if "what is" in question.lower():
            question_type = "definition"
        elif "how does" in question.lower() or "how do" in question.lower():
            question_type = "mechanism"
        elif "relationship" in question.lower() or "related" in question.lower():
            question_type = "relationship"
        elif "property" in question.lower() or "properties" in question.lower():
            question_type = "property"
        elif "classify" in question.lower() or "classification" in question.lower():
            question_type = "classification"
        else:
            question_type = "general"
        
        # Generate an answer based on question type and reasoning
        if question_type == "definition" and entities:
            entity = entities[0]["name"]
            ancestors = self.reasoner.get_ancestors(entities[0]["id"])[:3]
            answer = {
                "short": f"{entity} is a {' and '.join(ancestors)}.",
                "detailed": f"{entity} is classified as {', '.join(ancestors)}. " + 
                           (conclusion or "It has specific properties and relationships in the ChEBI ontology.")
            }
        
        elif question_type == "classification" and entities:
            entity = entities[0]["name"]
            ancestors = self.reasoner.get_ancestors(entities[0]["id"])[:5]
            answer = {
                "short": f"{entity} is classified as {', '.join(ancestors[:2])}.",
                "detailed": f"{entity} belongs to the following classifications: {', '.join(ancestors)}. " +
                           (conclusion or "")
            }
        
        elif question_type == "relationship" and len(entities) >= 2:
            entity1 = entities[0]["name"]
            entity2 = entities[1]["name"]
            
            # Find relationships in context
            relationships = reasoning_data["context"]["relationships"]
            if relationships:
                rel_str = ", ".join([f"{r['source']} {r['relation']} {r['target']}" for r in relationships[:2]])
                answer = {
                    "short": f"{entity1} and {entity2} are related: {rel_str}.",
                    "detailed": "The relationship between these entities includes: " + 
                               ", ".join([f"{r['source']} {r['relation']} {r['target']}" for r in relationships]) +
                               (f" {conclusion}" if conclusion else "")
                }
            else:
                # Find common classifications
                entity1_ancestors = set(self.reasoner.get_ancestors(entities[0]["id"]))
                entity2_ancestors = set(self.reasoner.get_ancestors(entities[1]["id"]))
                common = entity1_ancestors.intersection(entity2_ancestors)
                
                if common:
                    answer = {
                        "short": f"{entity1} and {entity2} share classifications: {', '.join(list(common)[:2])}.",
                        "detailed": f"While no direct relationship is recorded, {entity1} and {entity2} " +
                                  f"share these classifications: {', '.join(list(common)[:5])}." +
                                  (f" {conclusion}" if conclusion else "")
                    }
                else:
                    answer = {
                        "short": f"No direct relationship found between {entity1} and {entity2}.",
                        "detailed": f"The ChEBI ontology does not record a direct relationship between {entity1} and {entity2}."
                    }
        
        else:
            # General answer based on available reasoning
            if conclusion:
                answer = {
                    "short": conclusion.replace("Conclusion: ", ""),
                    "detailed": conclusion
                }
            else:
                # Use the last non-question step as a fallback
                for step in reversed(steps):
                    if "question:" not in step.lower():
                        answer = {
                            "short": step,
                            "detailed": step
                        }
                        break
                else:
                    answer = {
                        "short": "Unable to derive an answer from the available information.",
                        "detailed": "The ChEBI ontology does not contain sufficient information to answer this question."
                    }
        
        return answer
    
    def _estimate_confidence(self, reasoning: List[str], entities: List[Dict]) -> float:
        """
        Estimate the confidence in the reasoning and answer
        
        Args:
            reasoning: List of reasoning steps
            entities: List of entities identified
        
        Returns:
            Confidence score between 0.0 and 1.0
        """
        # Base confidence on several factors
        confidence = 0.5  # Start with neutral confidence
        
        # Factor 1: Were entities found?
        if not entities:
            return 0.2  # Very low confidence if no entities were found
        
        confidence += min(0.2, 0.05 * len(entities))  # Up to 0.2 boost for found entities
        
        # Factor 2: Entity coverage in question
        question_words = set(reasoning[0].lower().replace("question: ", "").split())
        entity_words = set()
        for entity in entities:
            entity_words.update(entity["name"].lower().split())
            entity_words.update(entity["original_mention"].lower().split())
        
        coverage = len(entity_words.intersection(question_words)) / max(1, len(question_words))
        confidence += 0.1 * coverage  # Up to 0.1 boost for coverage
        
        # Factor 3: Reasoning steps depth
        if len(reasoning) <= 2:  # Just question and one step
            confidence -= 0.1
        elif len(reasoning) >= 5:  # Detailed reasoning
            confidence += 0.1
        
        # Factor 4: Conclusion presence
        has_conclusion = any("conclusion" in step.lower() for step in reasoning)
        if has_conclusion:
            confidence += 0.1
        
        # Ensure confidence is between 0.0 and 1.0
        return max(0.0, min(1.0, confidence))
    
    def to_json(self, data: Dict) -> str:
        """Convert a reasoning or answer structure to JSON for LLM consumption"""
        return json.dumps(data, indent=2)
    
    def answer_batch(self, questions: List[str]) -> List[Dict]:
        """
        Process a batch of questions with reasoning
        
        Args:
            questions: List of questions to answer
        
        Returns:
            List of answer dictionaries with reasoning
        """
        return [self.answer_with_reasoning(q) for q in questions]


def demonstrate_llm_integration():
    """Demonstrate the LLM reasoning interface"""
    print("Initializing ChEBI LLM Reasoning Interface...")
    interface = ChEBILLMReasoningInterface(load_from_file=True)
    
    # Example questions
    questions = [
        "What is dopamine and how is it classified?",
        "What is the relationship between glucose and ATP?",
        "What properties does acetylcholine have?",
        "How are serotonin and dopamine related?",
        "What pathway involves glucose metabolism?"
    ]
    
    for question in questions:
        print(f"\n=== Answering: {question} ===")
        
        # Get answer with reasoning
        result = interface.answer_with_reasoning(question)
        
        # Display results
        print(f"Answer: {result['answer']['short']}")
        print(f"Confidence: {result['confidence']:.2f}")
        print("\nDetailed answer:")
        print(result['answer']['detailed'])
        
        print("\nReasoning steps:")
        for i, step in enumerate(result['reasoning']):
            print(f"  {i+1}. {step['step']}")
            print(f"     Type: {step['type']}")
            print(f"     Explanation: {step['explanation']}")
        
        print("\nKey context:")
        for entity_name, entity_data in result['context']['entities'].items():
            print(f"  - {entity_name}: {', '.join(entity_data['classifications'][:3])}")
        
        if result['context']['relationships']:
            print("  Relationships:")
            for rel in result['context']['relationships'][:3]:
                print(f"    - {rel['source']} {rel['relation']} {rel['target']}")
        
        print("\n" + "="*50)
    
    # Example of JSON output for LLM
    json_output = interface.to_json(result)
    print("\nJSON output for LLM consumption (example):")
    print(json_output[:500] + "... (truncated)")


if __name__ == "__main__":
    demonstrate_llm_integration()

ModuleNotFoundError: No module named 'chebi_reasoner'

In [20]:
import os
import json
import requests
from typing import Dict, List, Optional, Any



class ChEBILLMIntegration:
    """
    Integration between the ChEBI reasoner and an LLM API
    to provide accurate chemical reasoning capabilities.
    """
    
    def __init__(self, llm_api_endpoint: str = None, llm_api_key: str = None):
        """
        Initialize the integration with ChEBI reasoner and LLM
        
        Args:
            llm_api_endpoint: LLM API endpoint URL 
            llm_api_key: API key for the LLM service
        """
        # Initialize the reasoning interface
        self.reasoning_interface = ChEBILLMReasoningInterface(load_from_file=True)
        
        # LLM API configuration
        self.llm_api_endpoint = llm_api_endpoint or os.environ.get("LLM_API_ENDPOINT")
        self.llm_api_key = llm_api_key or os.environ.get("LLM_API_KEY")
        
        # Default prompt templates
        self.system_prompt_template = """
You are a chemical reasoning assistant with access to the ChEBI (Chemical Entities of Biological Interest) ontology.
You will receive a question along with structured reasoning derived from the ChEBI ontology.
Your task is to provide a clear, scientifically accurate answer that follows the reasoning provided.

Use the following approach:
1. Ensure your answer is consistent with the reasoning steps provided
2. Incorporate the contextual information about chemical entities
3. Express confidence levels appropriately based on the data provided
4. Cite relationships and classifications from ChEBI when relevant
5. Maintain scientific accuracy while being accessible to the user

The reasoning provided is based on ChEBI, a highly reliable scientific ontology for biochemical entities.
        """
        
        self.user_prompt_template = """
QUESTION: {question}

CHEMICAL REASONING:
{reasoning_summary}

ENTITY CONTEXT:
{entity_context}

Based on the chemical reasoning provided, please answer the question accurately and clearly.
Explain your reasoning process in a way that's scientifically sound but understandable.
If the provided reasoning indicates uncertainty, acknowledge the limitations in your answer.
"""
    
    def set_llm_api(self, endpoint: str, api_key: str):
        """Set the LLM API endpoint and key"""
        self.llm_api_endpoint = endpoint
        self.llm_api_key = api_key
    
    def set_prompt_templates(self, system_prompt: str = None, user_prompt: str = None):
        """Set custom prompt templates for the LLM"""
        if system_prompt:
            self.system_prompt_template = system_prompt
        if user_prompt:
            self.user_prompt_template = user_prompt
    
    def process_chemical_question(self, question: str, use_llm: bool = True) -> Dict:
        """
        Process a chemical question using ChEBI reasoning and optionally an LLM
        
        Args:
            question: The chemical question to process
            use_llm: Whether to use the LLM to generate the final answer
        
        Returns:
            Dictionary with the processed answer and reasoning
        """
        # Generate reasoning using the ChEBI ontology
        reasoning_result = self.reasoning_interface.answer_with_reasoning(question)
        
        if not use_llm or not self.llm_api_endpoint:
            # Return only the reasoning interface result if LLM is not used
            return {
                "question": question,
                "answer": reasoning_result["answer"]["detailed"],
                "confidence": reasoning_result["confidence"],
                "reasoning": [step["step"] for step in reasoning_result["reasoning"]],
                "source": "ChEBI Ontology"
            }
        
        # Prepare LLM prompt with reasoning
        llm_response = self._query_llm(reasoning_result)
        
        # Combine results
        combined_result = {
            "question": question,
            "answer": llm_response["content"] if llm_response else reasoning_result["answer"]["detailed"],
            "confidence": reasoning_result["confidence"],
            "reasoning": [step["step"] for step in reasoning_result["reasoning"]],
            "context": self._format_context_summary(reasoning_result["context"]),
            "source": "ChEBI Ontology + LLM" if llm_response else "ChEBI Ontology"
        }
        
        return combined_result
    
    def _query_llm(self, reasoning_result: Dict) -> Optional[Dict]:
        """
        Query the LLM API with the reasoning data
        
        Args:
            reasoning_result: The reasoning data from the ChEBI interface
        
        Returns:
            LLM response or None if the request failed
        """
        if not self.llm_api_endpoint or not self.llm_api_key:
            print("LLM API not configured. Set endpoint and API key first.")
            return None
        
        try:
            # Format the reasoning for the LLM
            reasoning_summary = self._format_reasoning_summary(reasoning_result["reasoning"])
            entity_context = self._format_entity_context(reasoning_result["context"])
            
            # Format the prompt
            user_prompt = self.user_prompt_template.format(
                question=reasoning_result["question"],
                reasoning_summary=reasoning_summary,
                entity_context=entity_context
            )
            
            # Prepare the API request
            headers = {
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.llm_api_key}"
            }
            
            payload = {
                "model": "gpt-4",  # Or another model identifier
                "messages": [
                    {"role": "system", "content": self.system_prompt_template},
                    {"role": "user", "content": user_prompt}
                ],
                "temperature": 0.2,  # Low temperature for more deterministic outputs
                "max_tokens": 1000
            }
            
            # Make the API request
            response = requests.post(
                self.llm_api_endpoint,
                headers=headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                if "choices" in result and len(result["choices"]) > 0:
                    return result["choices"][0]["message"]
            
            print(f"LLM API request failed with status code: {response.status_code}")
            print(f"Response: {response.text}")
            return None
        
        except Exception as e:
            print(f"Error querying LLM API: {e}")
            return None
    
    def _format_reasoning_summary(self, reasoning: List[Dict]) -> str:
        """Format reasoning steps for LLM consumption"""
        summary = ""
        for i, step in enumerate(reasoning):
            summary += f"{i+1}. {step['step']}\n"
            if step.get('explanation'):
                summary += f"   Explanation: {step['explanation']}\n"
        
        return summary
    
    def _format_entity_context(self, context: Dict) -> str:
        """Format entity context for LLM consumption"""
        formatted = "Known entities and their properties:\n"
        
        for entity_name, entity_data in context["entities"].items():
            formatted += f"\n- {entity_name}:\n"
            formatted += f"  Classifications: {', '.join(entity_data['classifications'][:3])}\n"
            
            if entity_data["properties"]:
                props = []
                for k, v in entity_data["properties"].items():
                    if not k.startswith('_'):
                        props.append(f"{k}: {v}")
                if props:
                    formatted += f"  Properties: {'; '.join(props[:3])}\n"
        
        if context["relationships"]:
            formatted += "\nRelationships between entities:\n"
            for rel in context["relationships"][:5]:
                formatted += f"- {rel['source']} {rel['relation']} {rel['target']}\n"
        
        return formatted
    
    def _format_context_summary(self, context: Dict) -> Dict:
        """Format context for the response"""
        return {
            "entities": list(context["entities"].keys()),
            "key_classifications": {
                name: data["classifications"][:3] 
                for name, data in context["entities"].items()
            },
            "relationships": [
                f"{rel['source']} {rel['relation']} {rel['target']}"
                for rel in context["relationships"][:3]
            ]
        }
    
    def save_response(self, result: Dict, filename: str = None):
        """Save a response to a JSON file"""
        if filename is None:
            # Generate a filename based on the question
            question_words = result["question"].lower().split()[:5]
            filename = f"{'_'.join(question_words)}.json"
        
        with open(filename, 'w') as f:
            json.dump(result, f, indent=2)
        
        print(f"Response saved to {filename}")


def demonstrate_llm_integration():
    """Demonstrate the LLM integration"""
    print("Initializing ChEBI LLM Integration...")
    
    # For the demonstration, we'll use a mock LLM response
    integration = ChEBILLMIntegration()
    
    # Example question
    question = "What is the relationship between dopamine and serotonin in neurotransmission?"
    
    print(f"\n=== Processing question: {question} ===\n")
    
    # First show reasoning without LLM
    print("Getting reasoning from ChEBI ontology...")
    result_without_llm = integration.process_chemical_question(question, use_llm=False)
    
    print("\nChEBI Reasoning steps:")
    for step in result_without_llm["reasoning"]:
        print(f"- {step}")
    
    print(f"\nConfidence: {result_without_llm['confidence']:.2f}")
    print("\nAnswer based on ChEBI only:")
    print(result_without_llm["answer"])
    
    print("\n" + "="*50)
    print("\nIn a production environment, the system would now:")
    print("1. Send this reasoning to an LLM API (like GPT-4)")
    print("2. The LLM would generate a coherent, scientifically accurate answer")
    print("3. The answer would incorporate the ChEBI reasoning steps")
    print("4. The final response would combine ontological accuracy with natural language fluency")
    
    # Mock what an LLM-enhanced answer might look like
    mock_llm_answer = """
Based on the ChEBI ontology analysis, dopamine and serotonin are both classified as neurotransmitters, specifically monoamine neurotransmitters. They share several functional similarities but have distinct roles in neurotransmission.

Dopamine is primarily involved in reward pathways, motor control, and executive functions. It's synthesized from tyrosine and functions through dopaminergic receptors.

Serotonin (5-hydroxytryptamine) is involved in regulating mood, appetite, sleep, and cognitive functions. It's synthesized from tryptophan and acts through serotonergic receptors.

While they don't directly interact in a chemical sense, these neurotransmitters often work in balance within the brain. For example, imbalances between dopamine and serotonin systems have been implicated in various neurological and psychiatric conditions. Several medications target both systems either directly or indirectly.

The ChEBI ontology confirms that both compounds share classifications as neurotransmitters and endogenous metabolites, supporting their related but distinct biological roles.
"""
    
    print("\nExample of what an LLM-enhanced answer might look like:")
    print(mock_llm_answer)


if __name__ == "__main__":
    demonstrate_llm_integration()

Initializing ChEBI LLM Integration...
Loading ChEBI ontology from chebi.obo...


Loading relationships: 100%|██████████| 374296/374296 [00:01<00:00, 292157.44it/s]


ChEBI ontology loaded successfully with 202139 nodes and 374296 edges.

=== Processing question: What is the relationship between dopamine and serotonin in neurotransmission? ===

Getting reasoning from ChEBI ontology...


AttributeError: 'list' object has no attribute 'items'